# Ozone (O₃) radiative-transfer model: Temperature dependence

This notebook models infrared absorption by ozone in the ν₃ band (1040–1050 cm⁻¹; 9.52–9.62 µm) and examines how the transmitted spectrum changes with gas temperature.

**Model conditions:** Temperature is swept from 190 K to 300 K. The ozone mole fraction (8 ppmv) and path length (300 m) are fixed. In the default constant-density mode, pressure varies with temperature according to the ideal-gas law, $P=nk_BT$.

**Data note:** Quantitative spectra require a real O₃ line list from HITRAN. Unit tests using synthetic lines test the implementation, not agreement with measured ozone spectra. The default atmospheric conditions are model assumptions, not measured snapshots.

## 1. Theory

**Fixed number density.** For an ideal gas, $n=P/(k_BT)$. At constant number density, $P=nk_BT\propto T$. The ozone number density is $n_{O_3}=xn$, where $x$ is the ozone mole fraction. With fixed path length $L$, the ozone column $N=n_{O_3}L$ remains constant during the sweep.

**Beer–Lambert law**

$$
\alpha(\tilde\nu)=n_{O_3}\sigma(\tilde\nu),\qquad
\sigma(\tilde\nu)=\sum_k S_k(T)\,\varphi_k(\tilde\nu-\tilde\nu_k),
$$
$$
\tau(\tilde\nu)=\alpha(\tilde\nu)L,\qquad
\mathcal T(\tilde\nu)=e^{-\tau(\tilde\nu)},\qquad
I(\tilde\nu)=I_0(\tilde\nu)\mathcal T(\tilde\nu).
$$

Here, $S_k(T)$ is the temperature-dependent line strength and $\varphi_k$ is the normalised line profile.

**Temperature-dependent quantities**

1. **Line strength.** HITRAN scales the reference line strength using the partition function and Boltzmann population:
$$
S(T)=S(T_{ref})\frac{Q(T_{ref})}{Q(T)}
\frac{e^{-c_2E''/T}}{e^{-c_2E''/T_{ref}}}
\frac{1-e^{-c_2\tilde\nu_0/T}}{1-e^{-c_2\tilde\nu_0/T_{ref}}},
$$
where $T_{ref}=296\,\mathrm K$, $c_2=hc/k_B$, and $E''$ is the lower-state energy. The temperature response depends on the lower-state energy and partition function.

2. **Doppler width.** The Gaussian half-width at half-maximum (HWHM) varies as
$$
\gamma_D=\frac{\tilde\nu_0}{c}
\sqrt{\frac{2k_BT\ln 2}{m}}\propto\sqrt T.
$$

3. **Lorentz width.** Collisional broadening is modelled as
$$
\gamma_L=\left(\frac{T_{ref}}{T}\right)^{n_{air}}
[\gamma_{air}(P-p_{self})+\gamma_{self}p_{self}].
$$
At constant number density, pressure increases with temperature, partly compensating for the temperature scaling of the broadening coefficient.

**Line shapes.** The Gaussian, Lorentzian and Voigt profiles are normalised so that
$\int\varphi(\tilde\nu)\,d\tilde\nu=1$. Their widths determine the line shape and peak height. The line-centre optical depth is
$$
\tau_0=n_{O_3}S\varphi(0)L.
$$

**Equivalent width.** The equivalent width is
$$
W=\int[1-\mathcal T(\tilde\nu)]\,d\tilde\nu.
$$
For optically thin lines, $W\approx SN$. Saturated lines also depend on line widths and wings.

**Model limitations.** The constant-density temperature sweep is an idealised experiment, not a vertical atmospheric profile. The model uses a uniform slab and does not include emission, scattering, other gases, continuum absorption or line mixing.

**What to examine:** Panel A compares the full-window spectrum at the reference and current temperatures. Panels B and C show the selected line's transmittance and optical depth, respectively.

## 2. HITRAN line-list data

The model requires an ozone line list saved in a `data/` folder beside this notebook.

1. Open [HITRAN](https://hitran.org/) and select **Data Access → Line-by-Line**.
2. Choose **O₃ (molecule 3)** and all isotopologues.
3. Request the wavenumber range **1015–1075 cm⁻¹**.
4. Include the parameters required by the model: line position, intensity, Einstein A, air/self-broadening parameters, lower-state energy, temperature exponent and pressure shift.
5. Download in `.par` format and save as `data/O3_1015_1075.par`.

The loader also accepts HAPI `.data` files. HAPI supplies isotopologue masses and partition sums. The optional download function requires a HITRAN account and a configured API key.

## 3. Setup

Install the required packages once, then restart the kernel:

```python
%pip install numpy scipy matplotlib ipywidgets hitran-api pillow
```

MP4 export additionally requires `ffmpeg`. HTML and GIF export do not.

In [1]:
import os, sys, json, glob, re, time, datetime
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import wofz

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    print("ipywidgets", widgets.__version__)
except ImportError:
    print("ipywidgets missing: run  %pip install -U ipywidgets  and restart the kernel (the static draw() fallback still works)")
try:
    import hapi
    print("HAPI", getattr(hapi, "__version__", "?"))
except ImportError:
    print("HAPI missing: run  %pip install hitran-api  (needed for partition sums and downloading)")
print("numpy", np.__version__)

ipywidgets 8.1.8
HAPI version: 1.3.0.0
To get the most up-to-date version please check http://hitran.org/hapi
ATTENTION: Python versions of partition sums from TIPS-2021 are now available in HAPI code

           MIT license: Copyright 2021 HITRAN team, see more at http://hitran.org. 

           If you use HAPI in your research or software development,
           please cite it using the following reference:
           R.V. Kochanov, I.E. Gordon, L.S. Rothman, P. Wcislo, C. Hill, J.S. Wilzewski,
           HITRAN Application Programming Interface (HAPI): A comprehensive approach
           to working with spectroscopic data, J. Quant. Spectrosc. Radiat. Transfer 177, 15-30 (2016)
           DOI: 10.1016/j.jqsrt.2016.03.005

           ATTENTION: This is the core version of the HITRAN Application Programming Interface.
                      For more efficient implementation of the absorption coefficient routine, 
                      as well as for new profiles, parameters and other f

In [2]:
class cfg:
    MODE = "T"                         # this notebook sweeps TEMPERATURE only
    # ---- spectral interval ----
    NU_MIN, NU_MAX = 1040.0, 1050.0     # cm^-1 (9.52-9.62 um), core of the nu3 band of O3
    NU_STEP   = 0.0002                  # cm^-1; must be well below the narrowest line width (warned automatically)
    WING_CM1  = 5.0                     # each line evaluated within +/- this (model choice)
    S_MIN     = 1.0e-26                 # drop lines weaker than this at 296 K [cm/molecule] (model choice)
    X_O3      = 8.0e-6                  # O3 mole fraction in the air (ASSUMED, order of the stratospheric peak)
    # ---- the temperature sweep ----
    T_RANGE      = (190.0, 300.0)       # K, the only quantity that is swept
    T_FIX        = 227.65               # K, REFERENCE temperature: the state at which the gas density is defined
    P_FIX_HPA    = 10.08                # hPa, TOTAL pressure at the reference temperature (US-1976, 31 km)
    P_COUPLING   = "const_density"      # "const_density": P = n k T with n fixed (fixed concentration; P rises with T)
                                        # "const_pressure": P fixed (n falls when T rises)
    L_FIX_M      = 300.0                # m, FIXED path length. tau0 of the strongest line is ~1 here (not saturated)
    N_FRAMES     = 61                   # frames of the sweep
    ANCHOR_DT_K  = 10.0                 # exact spectrum every <= this many K, cubic spline of ln(alpha) in between
    EXACT_ALL    = False                # True: compute every frame exactly (slower, no interpolation)
    SOURCE       = "flat"               # "flat" (I0 = 1), "sun" (5772 K), "lamp" (1500 K), "earth" (288 K)
    DATA_DIR     = "data"
    FETCH_MARGIN_CM1 = 25.0
    FIG_DPI_SCREEN = 80                 # figure on the screen
    FIG_DPI_SAVE   = 300                # saved PNG files
    # not used in this notebook (kept because shared code refers to them)
    L_RANGE_M    = (10.0, 100000.0)
    P_RANGE_HPA  = (1.0, 1013.0)

## 4. Physics engine

The following cells implement the absorption model and its numerical functions.

**Units:** wavenumber $\tilde\nu$: cm⁻¹; line strength $S$: cm/molecule; line profile $\varphi$: cm; cross-section $\sigma$: cm²/molecule; number density $n$: cm⁻³; absorption coefficient $\alpha$: cm⁻¹; path length $L$: cm. Line widths are HWHM.

In [3]:
"""
rt_core.py - physics engine for the ozone (O3) transmission model.

Everything here is plain NumPy/SciPy and does NOT need HITRAN data or HAPI,
so it can be unit-tested with synthetic lines (see tests/test_rt_core.py).

UNIT CONVENTIONS (HITRAN-style, CGS spectroscopy units)
  wavenumber nu ............ cm^-1
  line intensity S ......... cm^-1 / (molecule cm^-2)  == cm / molecule
  line-shape phi(nu) ....... cm            (integrates to 1 over cm^-1)
  cross section sigma ...... cm^2 / molecule   = S * phi
  number density n ......... molecule / cm^3
  absorption coeff alpha ... cm^-1         = n * sigma
  path length L ............ cm
  optical depth tau ........ dimensionless = alpha * L
  HWHM gamma ............... cm^-1  (HALF width at HALF maximum - NOT FWHM,
                                     NOT the 1/e half-width)
"""

from scipy.interpolate import CubicSpline

# NumPy 2.x renamed trapz -> trapezoid; support both.
trapezoid = getattr(np, "trapezoid", None) or np.trapz

# ----------------------------------------------------------------------------
# Constants (SI values are exact since the 2019 SI redefinition)
# ----------------------------------------------------------------------------
H_PLANCK = 6.62607015e-34      # J s
C_LIGHT = 299792458.0          # m / s
K_B = 1.380649e-23             # J / K
AMU = 1.66053906660e-27        # kg (atomic mass unit; CODATA 2018 value)
ATM_PA = 101325.0              # Pa per standard atmosphere
T_REF = 296.0                  # K, HITRAN reference temperature
# second radiation constant c2 = h c / k_B, converted from m K to cm K
C2_CM_K = H_PLANCK * C_LIGHT / K_B * 100.0
LN2 = np.log(2.0)


# ----------------------------------------------------------------------------
# Data container for one gas
# ----------------------------------------------------------------------------
@dataclass
class LineList:
    """HITRAN line parameters for one molecule (all isotopologues).

    S is the HITRAN intensity at T_REF = 296 K. HITRAN's `sw` column already
    includes the natural isotopic abundance, so n_species below must be the
    TOTAL molecule number density (do not multiply by abundance again).
    """
    name: str
    M: int                      # HITRAN molecule id (O3 = 3)
    nu: np.ndarray              # cm^-1
    S: np.ndarray               # cm^-1/(molecule cm^-2) at 296 K
    E_low: np.ndarray           # cm^-1, lower-state energy
    gamma_air: np.ndarray       # cm^-1/atm, HWHM at 296 K, 1 atm
    gamma_self: np.ndarray      # cm^-1/atm, HWHM at 296 K, 1 atm
    n_air: np.ndarray           # temperature exponent of gamma_air
    delta_air: np.ndarray       # cm^-1/atm, pressure shift
    iso: np.ndarray             # local isotopologue index I (int)
    mass_amu: dict              # I -> molecular mass in amu

    def select(self, mask):
        return LineList(self.name, self.M, self.nu[mask], self.S[mask],
                        self.E_low[mask], self.gamma_air[mask],
                        self.gamma_self[mask], self.n_air[mask],
                        self.delta_air[mask], self.iso[mask], self.mass_amu)


# ----------------------------------------------------------------------------
# Ideal-gas mixture
# ----------------------------------------------------------------------------
def number_density_cm3(T, P_pa):
    """Total molecule number density n = P / (k_B T), returned in cm^-3.

    Assumes an ideal gas (valid here: ~1e5 Pa or less, 200-300 K, where the
    second virial correction is small). Partial pressure of species i is
    p_i = x_i P (Dalton) and n_i = x_i n for an ideal-gas mixture.
    """
    return P_pa / (K_B * T) * 1.0e-6        # m^-3 -> cm^-3


def dilution_warnings(x, limit=0.01):
    """Air-broadening is used for all non-self collisions, which is only defensible
    if ozone is a trace gas. Return warning strings otherwise."""
    if x < 0 or x > 1:
        raise ValueError("mole fraction must be between 0 and 1")
    msgs = []
    if x > limit:
        msgs.append(
            "WARNING: x_O3 exceeds %.0f%%. The model uses HITRAN air-broadening "
            "for all non-self collisions and gamma_self only for the self part; "
            "results are an approximation and not validated in this regime." % (limit * 100))
    return msgs


# ----------------------------------------------------------------------------
# Temperature dependence of the line intensity (HITRAN definition)
#   S(T) = S(Tref) * [Q(Tref)/Q(T)]
#                  * exp(-c2 E''/T) / exp(-c2 E''/Tref)
#                  * [1 - exp(-c2 nu/T)] / [1 - exp(-c2 nu/Tref)]
# ----------------------------------------------------------------------------
def scale_intensity(S_ref, nu0, E_low, Q_ref_over_Q, T):
    """Scale HITRAN intensities from T_REF to T.

    Q_ref_over_Q : Q(T_REF)/Q(T), the partition-function ratio (per line)
    rB           : ratio of Boltzmann populations of the lower state
    rE           : ratio of stimulated-emission factors
    """
    rB = np.exp(-C2_CM_K * E_low * (1.0 / T - 1.0 / T_REF))
    rE = (-np.expm1(-C2_CM_K * nu0 / T)) / (-np.expm1(-C2_CM_K * nu0 / T_REF))
    return S_ref * Q_ref_over_Q * rB * rE


def partition_ratio_per_line(lines, T, partition_fn):
    """Q(T_REF)/Q(T) for every line (depends on the line's isotopologue).
    partition_fn(M, I, T) -> total internal partition sum (dimensionless)."""
    ratio = np.empty(len(lines.nu))
    for I in np.unique(lines.iso):
        r = partition_fn(lines.M, int(I), T_REF) / partition_fn(lines.M, int(I), T)
        ratio[lines.iso == I] = r
    return ratio


# ----------------------------------------------------------------------------
# Line widths
# ----------------------------------------------------------------------------
def doppler_hwhm(nu0, T, mass_amu):
    """Doppler HWHM [cm^-1] = (nu0/c) * sqrt(2 k_B T ln2 / m).
    (HWHM. The 1/e half-width is larger by 1/sqrt(ln2); FWHM is 2x HWHM.)"""
    m = np.asarray(mass_amu) * AMU
    return nu0 / C_LIGHT * np.sqrt(2.0 * K_B * T * LN2 / m)


def lorentz_hwhm(gamma_air, gamma_self, n_air, T, P_pa, x_self):
    """Collisional (Lorentz) HWHM [cm^-1] following the HITRAN definition:
         gamma_L = (Tref/T)^n_air * [ gamma_air (P - p_self) + gamma_self p_self ]
    with pressures in atm. 'Foreign' collisions are treated as air.
    The single exponent n_air is applied to the self part too (HITRAN
    provides no separate exponent in the standard list)."""
    P_atm = P_pa / ATM_PA
    p_self = x_self * P_atm
    return (T_REF / T) ** n_air * (gamma_air * (P_atm - p_self) + gamma_self * p_self)


# ----------------------------------------------------------------------------
# Voigt profile (Doppler Gaussian convolved with collisional Lorentzian)
# ----------------------------------------------------------------------------
def voigt_profile(dnu, gamma_D, gamma_L):
    """Normalised Voigt profile phi(dnu) [cm], with dnu = nu - nu0 [cm^-1].

    gamma_D, gamma_L are HWHM values. Gaussian sigma = gamma_D / sqrt(2 ln2).
    phi = Re[w(z)] / (sigma sqrt(2 pi)),   z = (dnu + i gamma_L) / (sigma sqrt 2)
    where w is the Faddeeva function (scipy.special.wofz).
    Integral of phi over all dnu is exactly 1.
    """
    sigma = gamma_D / np.sqrt(2.0 * LN2)
    z = (dnu + 1j * gamma_L) / (sigma * np.sqrt(2.0))
    return wofz(z).real / (sigma * np.sqrt(2.0 * np.pi))


# ----------------------------------------------------------------------------
# Cross section and absorption coefficient
# ----------------------------------------------------------------------------
def species_cross_section(grid, lines, T, P_pa, x_self, partition_fn,
                          wing_cm1=10.0, s_min=0.0):
    """sigma(nu) [cm^2/molecule] = sum_i S_i(T) * phi_i(nu), natural abundance
    included via HITRAN's S. Each line is evaluated only within +/- wing_cm1
    of its (pressure-shifted) centre: this truncation is a MODEL CHOICE
    (it drops the far Lorentzian wings and ignores sub-Lorentzian behaviour).
    """
    P_atm = P_pa / ATM_PA
    keep = ((lines.S >= s_min) & (lines.nu > grid[0] - wing_cm1)
            & (lines.nu < grid[-1] + wing_cm1))
    L = lines.select(keep)
    sigma = np.zeros_like(grid)
    if len(L.nu) == 0:
        return sigma
    S_T = scale_intensity(L.S, L.nu, L.E_low,
                          partition_ratio_per_line(L, T, partition_fn), T)
    mass = np.array([L.mass_amu[int(i)] for i in L.iso])
    nu_c = L.nu + L.delta_air * P_atm             # HITRAN pressure shift
    gD = doppler_hwhm(nu_c, T, mass)
    gL = lorentz_hwhm(L.gamma_air, L.gamma_self, L.n_air, T, P_pa, x_self)
    lo = np.searchsorted(grid, nu_c - wing_cm1)
    hi = np.searchsorted(grid, nu_c + wing_cm1)
    for k in range(len(nu_c)):
        if hi[k] > lo[k]:
            sigma[lo[k]:hi[k]] += S_T[k] * voigt_profile(
                grid[lo[k]:hi[k]] - nu_c[k], gD[k], gL[k])
    return sigma


def absorption_coefficient(grid, lines, T, P_pa, x, partition_fn, wing_cm1=10.0, s_min=0.0):
    """Absorption coefficient alpha [cm^-1] = n_O3 * sigma,  n_O3 = x * P/(k_B T)."""
    n_tot = number_density_cm3(T, P_pa)
    return x * n_tot * species_cross_section(
        grid, lines, T, P_pa, x, partition_fn, wing_cm1, s_min)


def column_DU(x, T, P_pa, L_m):
    """Ozone column N = x n L of a uniform slab, in Dobson Units (1 DU = 2.687e16 molecules/cm^2)."""
    return x * number_density_cm3(T, P_pa) * (L_m * 100.0) / 2.687e16


# ----------------------------------------------------------------------------
# Beer-Lambert transmission
# ----------------------------------------------------------------------------
def beer_lambert(alpha_total, L_cm, I0):
    """tau = alpha L ; transmittance = exp(-tau) ; I = I0 * transmittance.
    Valid for a uniform, non-scattering, non-emitting slab and radiation
    that is narrow compared with the line structure (monochromatic)."""
    tau = alpha_total * L_cm
    trans = np.exp(-tau)
    return tau, trans, I0 * trans


def integrated_tau(grid, alpha, L_cm):
    """Integral of tau(nu) d nu  [cm^-1]  (a spectrally integrated optical depth).
    NOT an absorbed energy: that would need I0(nu) weighting and saturation."""
    return trapezoid(alpha * L_cm, grid)


def equivalent_width(grid, trans):
    """W = integral of (1 - T(nu)) d nu  [cm^-1]; spectral width of a fully
    opaque band that would absorb the same amount of a FLAT incident spectrum."""
    return trapezoid(1.0 - trans, grid)


def line_strength_vs_T(lines, k, Ts, partition_fn):
    """S(T) [cm/molecule] for line index k at each temperature in Ts."""
    one = lines.select(np.arange(len(lines.nu)) == k)
    out = []
    for T in Ts:
        qr = partition_ratio_per_line(one, T, partition_fn)
        out.append(float(scale_intensity(one.S, one.nu, one.E_low, qr, T)[0]))
    return np.array(out)


# ----------------------------------------------------------------------------
# Incident spectrum
# ----------------------------------------------------------------------------
def planck_per_cm1(nu_cm1, T_K):
    """Planck spectral radiance per unit wavenumber, W m^-2 sr^-1 (cm^-1)^-1.
    B = 2 h c^2 nu^3 / (exp(h c nu / k T) - 1) with nu in m^-1, times
    100 m^-1 per cm^-1 to convert the per-wavenumber unit."""
    nu_m = np.asarray(nu_cm1) * 100.0
    x = H_PLANCK * C_LIGHT * nu_m / (K_B * T_K)
    return 2.0 * H_PLANCK * C_LIGHT ** 2 * nu_m ** 3 / np.expm1(x) * 100.0


# ----------------------------------------------------------------------------
# US Standard Atmosphere 1976, 0-32 km geopotential height (three layers)
#   0-11 km : T falls 6.5 K/km      11-20 km: isothermal 216.65 K      20-32 km: T rises 1 K/km
# ----------------------------------------------------------------------------
_G0, _M0, _RSTAR = 9.80665, 0.0289644, 8.31432

def _isa_layer_pressure(h, hb, Tb, Pb, lapse):
    if lapse == 0.0:
        return Tb + 0.0 * h, Pb * np.exp(-_G0 * _M0 * (h - hb) / (_RSTAR * Tb))
    T = Tb + lapse * (h - hb)
    return T, Pb * (Tb / T) ** (_G0 * _M0 / (_RSTAR * lapse))

_P11 = _isa_layer_pressure(11000.0, 0.0, 288.15, 101325.0, -0.0065)[1]
_P20 = _isa_layer_pressure(20000.0, 11000.0, 216.65, _P11, 0.0)[1]

def isa_atmosphere(h_m):
    """Temperature [K] and pressure [Pa] of the US-1976 standard atmosphere for 0 <= h <= 32 km."""
    scalar = np.ndim(h_m) == 0
    h = np.atleast_1d(np.asarray(h_m, dtype=float))
    if np.any(h < 0.0) or np.any(h > 32000.0 + 1e-9):
        raise ValueError("isa_atmosphere is implemented for 0-32 km only")
    T = np.empty_like(h); P = np.empty_like(h)
    for lo, hi, hb, Tb, Pb, lap in ((0.0, 11000.0, 0.0, 288.15, 101325.0, -0.0065),
                                    (11000.0, 20000.0, 11000.0, 216.65, _P11, 0.0),
                                    (20000.0, 32000.0, 20000.0, 216.65, _P20, 0.0010)):
        m = (h >= lo) & (h <= hi) if hi == 32000.0 else (h >= lo) & (h < hi)
        if m.any():
            T[m], P[m] = _isa_layer_pressure(h[m], hb, Tb, Pb, lap)
    return (T[0], P[0]) if scalar else (T, P)


# ----------------------------------------------------------------------------
# Many animation frames from few exact calculations
# ----------------------------------------------------------------------------
def choose_anchors(Ts, Ps, dT_max=10.0, dlnP_max=0.07):
    """Indices of frames that get an EXACT spectrum calculation. Frames must be
    sorted by increasing T. Consecutive anchors are at most dT_max kelvin apart
    and at most dlnP_max apart in ln(P) (matters when P varies along the sweep).
    The first and last frame are always anchors."""
    Ts, Ps = np.asarray(Ts, float), np.asarray(Ps, float)
    n = len(Ts)
    if n < 3:
        return np.arange(n)
    idx = [0]
    for i in range(1, n):
        j = idx[-1]
        if abs(Ts[i] - Ts[j]) > dT_max or abs(np.log(Ps[i] / Ps[j])) > dlnP_max:
            idx.append(i - 1 if i - 1 > j else i)
    if idx[-1] != n - 1:
        idx.append(n - 1)
    return np.array(sorted(set(idx)))


def interpolate_between_anchors(Ts, anchor_idx, anchor_values, rel_floor=1e-12):
    """Fill in the non-anchor frames by a cubic spline of ln(alpha) versus T.

    ln(alpha) is used because line strengths change roughly exponentially with
    temperature (Boltzmann factor), which a spline of ln(alpha) follows much
    better than a straight line in alpha. Values below rel_floor * max are
    treated as zero. This is an APPROXIMATION whose error must be checked
    (tests/test_rt_core.py and check_interpolation() in the notebook).
    anchor_values: array (n_anchor, n_grid). Returns array (len(Ts), n_grid)."""
    Ts = np.asarray(Ts, float)
    A = np.asarray(anchor_values, float)
    anchor_idx = np.asarray(anchor_idx)
    if len(anchor_idx) == len(Ts):
        return A.copy()
    peak = A.max()
    if peak <= 0:
        return np.zeros((len(Ts), A.shape[1]))
    floor = rel_floor * peak
    spline = CubicSpline(Ts[anchor_idx], np.log(np.maximum(A, floor)), axis=0)
    out = np.exp(spline(Ts))
    out[out < 2.0 * floor] = 0.0
    out[anchor_idx] = A                      # anchors stay exactly as computed
    return out


def envelope(x, y, n_bins=1500):
    """Reduce a long curve to 2*n_bins points (min and max of each bin) for
    plotting. Keeps every narrow peak and valley that a plain every-k-th-point
    thinning would miss. Output length is constant, so artists can be updated."""
    n = len(x)
    m = int(np.ceil(n / n_bins))
    if m <= 1:
        return np.asarray(x), np.asarray(y)
    pad = n_bins * m - n
    xp = np.pad(np.asarray(x, float), (0, pad), mode="edge").reshape(n_bins, m)
    yp = np.pad(np.asarray(y, float), (0, pad), mode="edge").reshape(n_bins, m)
    imin, imax = yp.argmin(axis=1), yp.argmax(axis=1)
    first, second = np.minimum(imin, imax), np.maximum(imin, imax)
    r = np.arange(n_bins)
    xs, ys = np.empty(2 * n_bins), np.empty(2 * n_bins)
    xs[0::2], ys[0::2] = xp[r, first], yp[r, first]
    xs[1::2], ys[1::2] = xp[r, second], yp[r, second]
    return xs, ys

### Line-shape functions

In [4]:
def gaussian_profile(dnu, gamma_D):
    # Normalised Doppler profile [cm]; gamma_D is the HWHM.
    return np.sqrt(LN2 / np.pi) / gamma_D * np.exp(-LN2 * (np.asarray(dnu) / gamma_D) ** 2)

def lorentz_profile(dnu, gamma_L):
    # Normalised Lorentzian [cm]; gamma_L is the HWHM.
    dnu = np.asarray(dnu)
    return gamma_L / np.pi / (dnu ** 2 + gamma_L ** 2)

def voigt_hwhm(gamma_D, gamma_L):
    # Approximate Voigt HWHM (Olivero & Longbothum 1977 fit, quoted accuracy ~0.02 %); inputs and output are HWHM.
    fL, fG = 2.0 * gamma_L, 2.0 * gamma_D
    return 0.5 * (0.5346 * fL + np.sqrt(0.2166 * fL ** 2 + fG ** 2))

def single_line_W(a_factor, gD, gL, L_cm_array):
    # Equivalent width of ONE line for each path length, for the three shapes.
    # a_factor = x * n * S(T) [cm^-1 per (cm) of profile]; the line is integrated over +/- h (finite window, a model choice).
    h = max(400.0 * gL, 40.0 * gD)
    d = np.concatenate([[0.0], np.geomspace(1e-4 * min(gD, gL), h, 6000)])
    shapes = {"Gaussian": gaussian_profile(d, gD), "Lorentzian": lorentz_profile(d, gL), "Voigt": voigt_profile(d, gD, gL)}
    return {name: np.array([2.0 * trapezoid(1.0 - np.exp(-a_factor * phi * L), d) for L in L_cm_array]) for name, phi in shapes.items()}

def step_warning(step, gV_min):
    # The line width that must be resolved is the Voigt HWHM (a tiny Lorentz width under a broad Doppler core does not matter).
    return None if step <= gV_min / 3.0 else ("WARNING: grid step %.2g cm-1 is larger than 1/3 of the narrowest Voigt HWHM (%.2g cm-1); lines are under-resolved. Reduce cfg.NU_STEP." % (step, gV_min))

## 5. HITRAN input and line-list loading

The loader reads HITRAN's fixed-width 160-character `.par` records and HAPI `.data` files. You can download the line list manually as described in Section 2 or use the optional HAPI download function.

No API key is stored in this notebook. If the HAPI download fails, use the manual download from [HITRAN](https://hitran.org/) and place the file in the `data/` folder.

In [5]:
"""
py - read HITRAN line lists and talk to HAPI.

* Lines are read by our own parser from HITRAN's fixed-width 160-character
  .par format, so they can come from HAPI's fetch (.data files in the same
  format) OR from a manual download on hitran.org (.par files).
* HAPI is used only for the isotopologue table (masses, abundances) and for
  the partition sums Q(T) (TIPS, computed locally, no internet needed).
"""



# 0-based slice positions of the fields we need in a 160-char .par line
_SLICES = dict(M=(0, 2), I=(2, 3), nu=(3, 15), S=(15, 25), A=(25, 35),
               gamma_air=(35, 40), gamma_self=(40, 45), E_low=(45, 55),
               n_air=(55, 59), delta_air=(59, 67))


def _to_float(txt):
    """Float parser tolerant of Fortran 'D' exponents and HITRAN's odd '2.700-164'."""
    t = txt.strip().replace("D", "E")
    try:
        return float(t)
    except ValueError:
        m = re.fullmatch(r"(\d\.\d{3})-(\d{3})", t)
        if m:
            return float(m.group(1) + "E-" + m.group(2))
        raise


def _iso_index(ch):
    """Local isotopologue id: '1'-'9', '0' -> 10, 'A' -> 11, 'B' -> 12 ..."""
    if ch == "0":
        return 10
    if ch.isdigit():
        return int(ch)
    return 11 + ord(ch) - ord("A")


def parse_par_files(paths, M_expected):
    """Parse .par / .data files into column arrays (dict)."""
    cols = {k: [] for k in _SLICES}
    cols["raw"] = []                                 # the complete HITRAN record (kept for duplicate detection and for quantum numbers)
    for path in paths:
        with open(path, "r") as fh:
            for line in fh:
                if len(line.rstrip("\n")) < 67:
                    continue
                if int(line[0:2]) != M_expected:
                    continue
                cols["raw"].append(line.rstrip("\n").rstrip())
                for key, (a, b) in _SLICES.items():
                    seg = line[a:b]
                    cols[key].append(_iso_index(seg) if key == "I"
                                     else int(seg) if key == "M" else _to_float(seg))
    return {k: np.array(v) for k, v in cols.items()}


def repair_missing(cols):
    """Documented policy for unusable broadening parameters (<= 0 or NaN).

    HITRAN uses placeholder values where a parameter is not available. Rather
    than invent numbers, missing values are replaced by DATA-DERIVED estimates
    (median of valid lines of the same isotopologue, else of the same gas),
    and the number of replaced lines is returned so it is reported openly.
    gamma_self falls back to the (repaired) gamma_air; because x_self is
    ~1e-4 the effect on the width is of order x_self * |gamma_self - gamma_air|.
    """
    report = {}
    for key in ("gamma_air", "n_air"):
        bad = ~(cols[key] > 0)
        report[key] = int(bad.sum())
        for I in np.unique(cols["I"][bad]) if bad.any() else []:
            sel = cols["I"] == I
            good = sel & ~bad
            ref = (np.median(cols[key][good]) if good.any()
                   else np.median(cols[key][~bad]))
            cols[key][sel & bad] = ref
    bad = ~(cols["gamma_self"] > 0)
    report["gamma_self"] = int(bad.sum())
    cols["gamma_self"][bad] = cols["gamma_air"][bad]
    return report


def iso_table(M):
    """{local I: (global_id, name, abundance, mass_amu)} from HAPI's ISO table."""
    import hapi
    out = {}
    for (m, I), v in hapi.ISO.items():
        if m == M:
            out[I] = (v[hapi.ISO_INDEX["id"]], v[hapi.ISO_INDEX["iso_name"]],
                      v[hapi.ISO_INDEX["abundance"]], v[hapi.ISO_INDEX["mass"]])
    return out


def hapi_partition_fn(M, I, T):
    """Total internal partition sum Q(M, I, T) from HAPI's built-in TIPS.
    (Which TIPS edition is used depends on the installed HAPI version.)"""
    import hapi
    q = hapi.partitionSum(M, I, float(T))
    if isinstance(q, tuple):
        q = q[1]
    return float(np.atleast_1d(q)[0])


def load_linelist(name, M, data_dir="data"):
    """Load all HITRAN files for one gas. Looks for <name>*.data and <name>*.par."""
    # BUG FIX: only files that belong to THIS gas (name*.par / name*.data). The old version read every
    # *.par, *.data, *.txt and *.out file, so one download saved in 2-3 formats was counted 2-3 times.
    paths = sorted(glob.glob(os.path.join(data_dir, name + "*.data")) + glob.glob(os.path.join(data_dir, name + "*.par")))
    if not paths:
        raise FileNotFoundError(
            "No %s line list in '%s'. Run fetch_hitran.py, or download a .par "
            "file from hitran.org and name it e.g. %s_2155_2425.par."
            % (name, data_dir, name))
    cols = parse_par_files(paths, M)
    if len(cols["nu"]) == 0:
        raise ValueError("Files %s contain no lines for molecule id %d" % (paths, M))
    # Duplicates = identical COMPLETE records (the same download stored in two files, e.g. O3.par and O3_1015_1075.par).
    # Test 4 compared only (iso, nu, S, E'', gamma_air) and wrongly removed genuine lines that differ in their quantum numbers.
    _, first = np.unique(cols["raw"], return_index=True)
    first = np.sort(first)
    n_dup = len(cols["nu"]) - len(first)
    for k in cols: cols[k] = cols[k][first]
    if n_dup: print("load_linelist: removed %d exact duplicate records (files: %s)" % (n_dup, [os.path.basename(p) for p in paths]))
    report = repair_missing(cols)
    report["duplicate_lines_removed"] = int(n_dup)
    masses = {I: v[3] for I, v in iso_table(M).items()}
    lines = LineList(name, M, cols["nu"], cols["S"], cols["E_low"],
                     cols["gamma_air"], cols["gamma_self"], cols["n_air"],
                     cols["delta_air"], cols["I"].astype(int), masses)
    lines.raw = cols["raw"]                           # full HITRAN record of every line (same order as lines.nu)
    return lines, report, paths


def show_assignment(nu, lines=None, n=1):
    """Print the HITRAN quantum assignment of the line(s) nearest to `nu` (cm-1).
    Fields of the 160-character record: V' (cols 68-82), V'' (83-97) = global (vibrational) quanta of the upper and
    lower state; Q' (98-112), Q'' (113-127) = local (rotational) quanta, e.g. J Ka Kc and the branch symbol."""
    L = O3 if lines is None else lines
    for k in np.argsort(np.abs(L.nu - nu))[:n]:
        r = L.raw[k].ljust(160)
        print("line %.6f cm-1   S(296 K) = %.3e   E'' = %.4f cm-1   E' = E'' + nu = %.4f cm-1   isotopologue %d" % (L.nu[k], L.S[k], L.E_low[k], L.E_low[k] + L.nu[k], L.iso[k] if hasattr(L, "iso") else -1))
        print("   upper vib. V'  = '%s'    lower vib. V'' = '%s'" % (r[67:82].strip(), r[82:97].strip()))
        print("   upper rot. Q'  = '%s'    lower rot. Q'' = '%s'" % (r[97:112].strip(), r[112:127].strip()))

In [6]:
MOLECULES = (("O3", 3),)                   # HITRAN molecule ids

def fetch_hitran():
    # Download the O3 line list with HAPI and write data/data_manifest.json.
    try:
        import hapi
    except ImportError:
        raise RuntimeError("HAPI not installed. Run: %pip install hitran-api")
    os.makedirs(cfg.DATA_DIR, exist_ok=True)
    hapi.db_begin(cfg.DATA_DIR)
    numin, numax = cfg.NU_MIN - cfg.FETCH_MARGIN_CM1, cfg.NU_MAX + cfg.FETCH_MARGIN_CM1
    hapi.VARIABLES["DISPLAY_FETCH_URL"] = True
    manifest = {"retrieval_date": datetime.date.today().isoformat(), "source": "HITRANonline via HAPI",
                "hapi_version": getattr(hapi, "__version__", "unknown"),
                "wavenumber_range_cm-1": [numin, numax],
                "units": {"nu": "cm-1", "S": "cm-1/(molecule cm-2) at 296 K, abundance-weighted",
                          "gamma": "cm-1/atm HWHM at 296 K", "E_low": "cm-1"}, "gases": {}}
    for name, M in MOLECULES:
        iso = iso_table(M)
        table = "%s_%d_%d" % (name, round(numin), round(numax))
        try:
            hapi.fetch_by_ids(table, [v[0] for v in iso.values()], numin, numax)
        except Exception as exc:
            raise RuntimeError("Download failed for %s: %s\nUse the manual .par route described above." % (name, exc))
        manifest["gases"][name] = {"hitran_molecule_id": M, "table": table,
            "isotopologues": {str(I): {"global_id": v[0], "name": v[1], "abundance": v[2], "mass_amu": v[3]}
                              for I, v in iso.items()}}
    for name, M in MOLECULES:
        lines, report, paths = load_linelist(name, M, cfg.DATA_DIR)
        manifest["gases"][name].update(n_lines=int(len(lines.nu)), files=[os.path.basename(p) for p in paths],
                                       parameters_replaced_by_data_derived_estimate=report)
        print("%s: %d lines; replaced parameters: %s" % (name, len(lines.nu), report))
    with open(os.path.join(cfg.DATA_DIR, "data_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=2)
    print("Wrote", os.path.join(cfg.DATA_DIR, "data_manifest.json"))

## 6. Tests

The tests use synthetic spectral lines and a toy partition function. They check the equations and numerical implementation; they do not validate the model against measured ozone spectra.

In [7]:
def toy_Q(M, I, T):
    return T


def make_lines(nu, S, E=0.0, g_air=0.07, g_self=0.07, n=0.7, delta=0.0, M=3, mass=48.0):
    nu = np.atleast_1d(np.asarray(nu, float))
    k = len(nu)
    full = lambda v: np.full(k, v, float) if np.isscalar(v) else np.asarray(v, float)
    return LineList("toy", M, nu, full(S), full(E), full(g_air), full(g_self),
                       full(n), full(delta), np.ones(k, int), {1: mass})


# ---------------------------------------------------------------- line shape
def _norm_with_tail(gD, gL, W, step):
    x = np.arange(-W, W + step, step)
    area = trapezoid(voigt_profile(x, gD, gL), x)
    return area, area + 2 * gL / (np.pi * W)       # + analytic Lorentzian tail


def test_voigt_normalisation():
    # Raw integral over +/-W misses the Lorentzian tail ~ 2 gL/(pi W);
    # adding that tail analytically must give 1 to high accuracy.
    for gD, gL in [(0.002, 0.04), (0.01, 0.001), (0.01, 0.0), (0.002, 0.2)]:
        raw, corrected = _norm_with_tail(gD, gL, W=2000.0, step=min(gD, max(gL, 1e-3)) / 8)
        assert abs(corrected - 1.0) < 2e-5, (gD, gL, raw, corrected)
        assert raw <= 1.0 + 1e-9


def test_voigt_limits_and_hwhm_convention():
    # pure Gaussian: peak 1/(sigma sqrt(2 pi)); value at dnu = gD must be half the peak (HWHM!)
    gD = 0.01
    peak = voigt_profile(0.0, gD, 0.0)
    assert abs(voigt_profile(gD, gD, 0.0) / peak - 0.5) < 1e-12
    sigma = gD / np.sqrt(2 * np.log(2))
    assert abs(peak - 1 / (sigma * np.sqrt(2 * np.pi))) < 1e-9 * peak
    # pure Lorentzian: peak 1/(pi gL); half maximum at dnu = gL
    gL = 0.05
    p = voigt_profile(0.0, 1e-7, gL)
    assert abs(p - 1 / (np.pi * gL)) / p < 1e-4
    assert abs(voigt_profile(gL, 1e-7, gL) / p - 0.5) < 1e-4


def test_doppler_width_scaling_and_value():
    a = doppler_hwhm(1042.0, 250.0, 48.0)
    b = doppler_hwhm(1042.0, 1000.0, 48.0)
    assert abs(b / a - 2.0) < 1e-12                       # proportional to sqrt(T)
    ref = 1042.0 / C_LIGHT * np.sqrt(2 * 1.380649e-23 * 250 * np.log(2) / (48.0 * 1.66053906660e-27))
    assert abs(a - ref) / ref < 1e-12


# ------------------------------------------------------- intensity scaling
def test_intensity_scaling_identity_and_formula():
    S = scale_intensity(1e-20, 2300.0, 500.0, 1.0, T_REF)
    assert abs(S - 1e-20) < 1e-30                          # T = T_ref -> unchanged
    T, nu, E, S0 = 230.0, 2224.0, 300.0, 2e-19
    c2 = 1.438776877
    expected = (S0 * (T_REF / T) * np.exp(-c2 * E / T) / np.exp(-c2 * E / T_REF)
                * (1 - np.exp(-c2 * nu / T)) / (1 - np.exp(-c2 * nu / T_REF)))
    got = scale_intensity(S0, nu, E, toy_Q(0, 0, T_REF) / toy_Q(0, 0, T), T)
    assert abs(got - expected) / expected < 1e-8


def test_c2_value():
    assert abs(C2_CM_K - 1.438776877) < 1e-8


def test_temperature_trends_differ_between_transitions():
    # Low-E'' line weakens on heating, high-E'' line strengthens: no universal trend.
    low = [scale_intensity(1e-19, 2300.0, 0.0, T_REF / T, T) for T in (250, 296, 400)]
    high = [scale_intensity(1e-19, 2300.0, 1500.0, T_REF / T, T) for T in (250, 296, 400)]
    assert low[0] > low[1] > low[2]
    assert high[0] < high[1] < high[2]


def test_stimulated_emission_rayleigh_jeans_limit():
    rE = scale_intensity(1.0, 1e-4, 0.0, 1.0, 200.0)   # nu -> 0: ratio -> T_ref/T
    assert abs(rE - T_REF / 200.0) < 1e-6


# -------------------------------------------------------------- broadening
def test_pressure_and_temperature_dependence_of_lorentz_width():
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.0)
    assert abs(g - 0.07) < 1e-12                           # 1 atm, 296 K, no self
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 0.5 * 101325.0, 0.0)
    assert abs(g - 0.035) < 1e-12                          # linear in P
    g = lorentz_hwhm(0.07, 0.09, 0.7, 200.0, 101325.0, 0.0)
    assert abs(g - 0.07 * (296 / 200) ** 0.7) < 1e-12
    g = lorentz_hwhm(0.07, 0.09, 0.7, 296.0, 101325.0, 0.1)   # self part mixed in
    assert abs(g - (0.07 * 0.9 + 0.09 * 0.1)) < 1e-12


# ------------------------------------------------- units / sum rule / gas law
def test_loschmidt_number_density():
    n = number_density_cm3(273.15, 101325.0) * 1e6      # back to m^-3
    assert abs(n - 2.686780111e25) / 2.686780111e25 < 1e-6


def test_line_strength_sum_rule():
    # integral of sigma d nu over an isolated line must equal S(T) [cm/molecule]
    nu0, S0, W = 2300.0, 1e-18, 400.0
    grid = np.arange(nu0 - W, nu0 + W, 0.005)
    lines = make_lines([nu0], [S0], E=200.0)
    sig = species_cross_section(grid, lines, 250.0, 0.5 * 101325, 0.0, toy_Q, wing_cm1=W)
    S_T = scale_intensity(S0, nu0, 200.0, T_REF / 250.0, 250.0)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, 250.0, 0.5 * 101325, 0.0)
    area = trapezoid(sig, grid) + S_T * 2 * gL / (np.pi * W)
    assert abs(area - S_T) / S_T < 2e-5


# -------------------------------------------------------- Beer-Lambert etc.
def _toy_gas(x=8e-6, T=221.55, P=2549.0, **kw):
    grid = np.arange(1040.0, 1050.0, 0.001)
    lines = make_lines([1042.0, 1045.5, 1047.2], [3e-20, 1e-20, 2e-20], E=[50, 300, 900])
    a = absorption_coefficient(grid, lines, T, P, x, toy_Q, wing_cm1=5.0, **kw)
    return grid, a


def test_beer_lambert_consistency_and_bounds():
    grid, a = _toy_gas()
    I0 = np.linspace(1.0, 2.0, len(grid))
    tau, T, I = beer_lambert(a, 1.0e5, I0)
    assert np.allclose(T, np.exp(-a * 1.0e5))
    assert np.all((T >= 0) & (T <= 1))
    assert np.allclose(I, I0 * T)
    m = T > 0
    assert np.allclose(tau[m], -np.log(T[m]), rtol=1e-9, atol=1e-12)


def test_zero_absorber_gives_unit_transmission():
    grid, a = _toy_gas(x=0.0)
    _, T, _ = beer_lambert(a, 1e8, np.ones_like(grid))
    assert np.all(T == 1.0)


def test_low_abundance_and_short_path_limit():
    grid, a = _toy_gas(x=1e-14)
    tau, T, _ = beer_lambert(a, 1.0, np.ones_like(grid))
    assert np.max(np.abs((1 - T) - tau)) < 1e-12           # 1 - exp(-tau) ~ tau


def test_optically_thick_limit():
    grid, a = _toy_gas(x=1e-3)
    _, T, _ = beer_lambert(a, 1.0e8, np.ones_like(grid))
    assert T.min() < 1e-10


def test_linearity_in_mole_fraction_and_layers_multiply():
    # with gamma_self == gamma_air the widths do not depend on x, so alpha is exactly linear in x
    g1, a1 = _toy_gas(x=4e-6)
    g2, a2 = _toy_gas(x=8e-6)
    assert np.allclose(2 * a1, a2, rtol=1e-12)
    # two slabs in a row: transmittances multiply, optical depths add
    _, T1, _ = beer_lambert(a2, 3.0e4, np.ones_like(g2))
    _, T2, _ = beer_lambert(a2, 7.0e4, np.ones_like(g2))
    _, T12, _ = beer_lambert(a2, 1.0e5, np.ones_like(g2))
    assert np.allclose(T1 * T2, T12, rtol=1e-12)


def test_ozone_column_in_dobson_units():
    # 8 ppmv in a 12 km slab at the 25 km standard-atmosphere conditions is about 300 DU (a typical total column)
    T, P = isa_atmosphere(25000.0)
    du = column_DU(8e-6, T, P, 12000.0)
    assert 270.0 < du < 330.0, du
    # and the conversion is linear in path length and in x
    assert abs(column_DU(8e-6, T, P, 24000.0) / du - 2.0) < 1e-12
    assert abs(column_DU(4e-6, T, P, 12000.0) / du - 0.5) < 1e-12


# --------------------------------------------------------- resolution tests
def test_resolution_convergence():
    rng = np.random.default_rng(1)
    nu = np.sort(rng.uniform(2295, 2305, 25))
    lines = make_lines(nu, rng.uniform(1e-19, 3e-18, 25), E=rng.uniform(0, 1500, 25), M=2, mass=44.0)

    def integrated(step):
        grid = np.arange(2290.0, 2310.0, step)
        sig = species_cross_section(grid, lines, 255.0, 54020.0, 4e-4, toy_Q, wing_cm1=10.0)
        return trapezoid(sig, grid)
    ref = integrated(0.00125)
    err_fine = abs(integrated(0.005) - ref) / ref
    err_coarse = abs(integrated(0.1) - ref) / ref
    assert err_fine < 1e-4
    assert err_coarse > err_fine                           # error shrinks as resolution improves


# ----------------------------------------------------------- source and ISA
def test_planck_peak_follows_wien_law_in_wavenumber():
    T = 1500.0
    nu = np.arange(500.0, 8000.0, 0.5)
    peak = nu[np.argmax(planck_per_cm1(nu, T))]
    assert abs(peak - 2.8214393721 * T / C2_CM_K) < 1.0


def test_isa_matches_tabulated_values():
    # US Standard Atmosphere 1976 table values
    T0, P0 = isa_atmosphere(0.0)
    assert abs(T0 - 288.15) < 1e-12 and abs(P0 - 101325.0) < 1e-9
    T, P = isa_atmosphere(11000.0)
    assert abs(T - 216.65) < 1e-9 and abs(P - 22632.1) / 22632.1 < 1e-3
    T, P = isa_atmosphere(20000.0)
    assert abs(T - 216.65) < 1e-9 and abs(P - 5474.9) / 5474.9 < 1e-3
    T, P = isa_atmosphere(32000.0)
    assert abs(T - 228.65) < 1e-9 and abs(P - 868.02) / 868.02 < 1e-3
    T, P = isa_atmosphere(25000.0)
    assert abs(T - 221.65) < 1e-9 and 2400.0 < P < 2700.0


def test_isa_20_32km_is_monotonic_and_vectorised():
    h = np.linspace(20000.0, 32000.0, 101)
    T, P = isa_atmosphere(h)
    assert np.all(np.diff(T) > 0) and np.all(np.diff(P) < 0)
    assert T.shape == h.shape and np.ndim(isa_atmosphere(25000.0)[0]) == 0
    try:
        isa_atmosphere(40000.0)
        raise AssertionError("should have refused 40 km")
    except ValueError:
        pass


def test_dilution_warning():
    assert dilution_warnings(8e-6) == []
    assert len(dilution_warnings(0.2)) == 1


def test_stratospheric_doppler_and_lorentz_widths_are_comparable():
    # at 25 km both widths are of the same order near 1042 cm-1; at the surface Lorentz dominates
    T, P = isa_atmosphere(25000.0)
    gD = doppler_hwhm(1042.0, T, 48.0)
    gL = lorentz_hwhm(0.07, 0.07, 0.7, T, P, 0.0)
    assert 0.2 < gL / gD < 20.0
    T0, P0 = isa_atmosphere(0.0)
    assert lorentz_hwhm(0.07, 0.07, 0.7, T0, P0, 0.0) / doppler_hwhm(1042.0, T0, 48.0) > 50.0


# ------------------------------------------- many frames from few exact ones
def _hard_lines(seed=3, n=150):
    # deliberately hard: huge spread of lower-state energies (up to 3000 cm^-1) and strengths
    rng = np.random.default_rng(seed)
    nu = np.sort(rng.uniform(2255.0, 2345.0, n))
    f = lambda v: np.full(n, v)
    return LineList("toy", 3, nu, 10 ** rng.uniform(-22, -18, n), rng.uniform(0, 3000, n),
                       rng.uniform(0.05, 0.09, n), f(0.09), rng.uniform(0.6, 0.8, n), f(-0.003),
                       np.ones(n, int), {1: 48.0})


def _exact_vs_interpolated(Ts, Ps):
    lines = _hard_lines()
    Q = lambda M, I, T: T ** 1.5
    grid = np.arange(2270.0, 2330.0, 0.005)
    one = lambda T, P: species_cross_section(grid, lines, T, P, 8e-6, Q, wing_cm1=10.0) * 8e-6 * number_density_cm3(T, P)
    exact = np.array([one(T, P) for T, P in zip(Ts, Ps)])
    anchors = choose_anchors(Ts, Ps)
    interp = interpolate_between_anchors(Ts, anchors, exact[anchors])
    core = exact > 1e-3 * exact.max()
    rel = np.max(np.abs(interp - exact)[core] / exact[core])
    dT = np.max(np.abs(np.exp(-interp * 1e4) - np.exp(-exact * 1e4)))      # 100 m path
    return anchors, rel, dT


def test_choose_anchors_spacing_and_ends():
    Ts = np.linspace(216.0, 316.0, 101)
    a = choose_anchors(Ts, np.full(101, 5e4))
    assert a[0] == 0 and a[-1] == 100
    assert np.max(np.diff(Ts[a])) <= 10.0 + 1e-9 and len(a) == 11
    Ts2, Ps2 = isa_atmosphere(np.linspace(20000, 32000, 101))
    o = np.argsort(Ts2)
    a2 = choose_anchors(Ts2[o], Ps2[o])
    assert np.max(np.abs(np.diff(np.log(Ps2[o][a2])))) <= 0.07 + 1e-9      # pressure spacing respected


def test_interpolated_frames_accuracy_fixed_pressure():
    Ts = np.linspace(216.0, 316.0, 101)
    anchors, rel, dT = _exact_vs_interpolated(Ts, np.full(101, 54019.9))
    assert len(anchors) == 11
    assert rel < 5e-3 and dT < 2e-3, (rel, dT)


def test_interpolated_frames_accuracy_isa_path():
    Ts, Ps = isa_atmosphere(np.linspace(20000, 32000, 101))
    o = np.argsort(Ts)
    anchors, rel, dT = _exact_vs_interpolated(Ts[o], Ps[o])
    assert rel < 1e-2 and dT < 4e-3, (rel, dT)   # residual dominated by pressure-shift/cut-off-edge effects


def test_anchor_frames_are_returned_unchanged():
    Ts = np.linspace(216.0, 316.0, 21)
    A = np.abs(np.random.default_rng(0).normal(size=(3, 50))) + 1e-3
    out = interpolate_between_anchors(Ts, np.array([0, 10, 20]), A)
    assert np.array_equal(out[[0, 10, 20]], A)
    assert np.all(out > 0)


def test_envelope_keeps_extremes_and_length():
    x = np.linspace(0.0, 10.0, 100000)
    y = np.sin(37 * x) * np.exp(-0.1 * x)
    xs, ys = envelope(x, y, 1500)
    assert len(xs) == 3000 and len(ys) == 3000
    assert ys.min() == y.min() and ys.max() == y.max()
    assert np.all(np.diff(xs) >= 0)

In [8]:
def test_profiles_are_normalised_and_have_the_right_hwhm():
    pos = np.geomspace(1e-7, 2000.0, 400001)               # dense near the centre, wide wings
    d = np.concatenate([-pos[::-1], [0.0], pos])
    for gD, gL in ((1e-3, 1e-3), (8e-4, 8e-2), (5e-3, 1e-4)):
        assert abs(trapezoid(gaussian_profile(d, gD), d) - 1) < 1e-6
        assert abs(trapezoid(lorentz_profile(d, gL), d) - 1) < 5e-4          # finite window misses a ~2 gamma/(pi W) tail
    assert abs(gaussian_profile(0.7e-3, 0.7e-3) / gaussian_profile(0.0, 0.7e-3) - 0.5) < 1e-12
    assert abs(lorentz_profile(0.01, 0.01) / lorentz_profile(0.0, 0.01) - 0.5) < 1e-12

def test_voigt_reduces_to_gaussian_and_lorentzian():
    d = np.linspace(-0.01, 0.01, 2001)
    g = voigt_profile(d, 1e-3, 1e-9); assert np.max(np.abs(g - gaussian_profile(d, 1e-3))) / g.max() < 1e-5
    d = np.linspace(-1.0, 1.0, 4001)
    l = voigt_profile(d, 1e-6, 1e-2); assert np.max(np.abs(l - lorentz_profile(d, 1e-2))) / l.max() < 1e-3

def test_voigt_hwhm_fit_against_numerical_half_maximum():
    for gD, gL in ((1e-3, 1e-3), (1e-3, 1e-2), (1e-2, 1e-3), (1e-3, 0.0)):
        d = np.linspace(0.0, 0.5, 2000001)
        v = voigt_profile(d, gD, max(gL, 1e-12))
        half = d[np.argmax(v < 0.5 * v[0])]
        assert abs(half - voigt_hwhm(gD, gL)) / half < 2e-3, (gD, gL, half, voigt_hwhm(gD, gL))

def test_curve_of_growth_thin_and_saturated():
    gD, gL = 8e-4, 2e-3
    S_N_small = 1e-6                                  # tau_0 << 1: W ~ integrated tau = a * L (a_factor*L)
    W = single_line_W(1.0, gD, gL, [S_N_small])
    for name in W: assert abs(W[name][0] - S_N_small) / S_N_small < 5e-2, name
    big = single_line_W(1.0, gD, gL, [1.0, 10.0])     # S N = 1 and 10 (tau_0 >> 1), window +/-0.8 cm-1 still wide enough
    ratio = big["Lorentzian"][1] / big["Lorentzian"][0]
    assert 2.7 < ratio < 3.4, ratio                    # Lorentzian: W ~ 2 sqrt(S N gamma_L) -> x sqrt(10)
    assert abs(big["Lorentzian"][1] / (2 * np.sqrt(10.0 * gL)) - 1) < 0.12
    assert big["Gaussian"][1] / big["Gaussian"][0] < 1.6               # Gaussian saturates much more slowly
    assert big["Lorentzian"][1] > big["Gaussian"][1]

def test_step_warning_logic():
    assert step_warning(1e-4, 8e-4) is None
    assert step_warning(5e-4, 8e-4) is not None

def test_loader_keeps_distinct_lines_and_drops_file_copies():
    import tempfile
    def rec(nu, S, quanta):
        base = "%2d%1d%12.6f%10.3E%10.3E%5.3f%5.3f%10.4f%4.2f%8.6f" % (3, 1, nu, S, 1.0e-2, 0.075, 0.07, 40.0, 0.7, -0.0002)
        return (base.ljust(67) + quanta.ljust(93))[:160]
    a = rec(1045.000000, 2.0e-20, "  0 0 1 0 0 0   0 0 0 0 0 0   10  2  8   10  3  7")
    b = rec(1045.000000, 2.0e-20, "  0 0 1 0 0 0   0 0 0 0 0 0   10  3  8   10  2  9")   # same 5 numbers, different quantum numbers: a genuine second line
    c = rec(1046.000000, 1.0e-21, "  0 0 1 0 0 0   0 0 0 0 0 0    9  1  9    9  2  8")
    with tempfile.TemporaryDirectory() as d:
        for fn in ("O3_one.par", "O3_copy.par"):                          # the same download saved twice
            with open(os.path.join(d, fn), "w") as fh: fh.write("\n".join([a, b, c]) + "\n")
        with open(os.path.join(d, "notes.txt"), "w") as fh: fh.write("\n".join([a, b, c]) + "\n")   # must be ignored
        lines, rep, _ = load_linelist("O3", 3, d)
    assert len(lines.nu) == 3, len(lines.nu)                              # a, b, c kept once each
    assert rep["duplicate_lines_removed"] == 3


In [9]:
def run_tests():
    names = sorted(n for n, f in globals().items() if n.startswith("test_") and callable(f))
    failed = 0
    for n in names:
        try:
            globals()[n]()
            print("PASS", n)
        except Exception as exc:
            failed += 1
            print("FAIL", n, "->", repr(exc))
    print("\n%d/%d tests passed (synthetic data)" % (len(names) - failed, len(names)))

run_tests()

PASS test_anchor_frames_are_returned_unchanged
PASS test_beer_lambert_consistency_and_bounds
PASS test_c2_value
PASS test_choose_anchors_spacing_and_ends
PASS test_curve_of_growth_thin_and_saturated
PASS test_dilution_warning
PASS test_doppler_width_scaling_and_value
PASS test_envelope_keeps_extremes_and_length
PASS test_intensity_scaling_identity_and_formula
PASS test_interpolated_frames_accuracy_fixed_pressure
PASS test_interpolated_frames_accuracy_isa_path
PASS test_isa_20_32km_is_monotonic_and_vectorised
PASS test_isa_matches_tabulated_values
PASS test_line_strength_sum_rule
PASS test_linearity_in_mole_fraction_and_layers_multiply
load_linelist: removed 3 exact duplicate records (files: ['O3_copy.par', 'O3_one.par'])
PASS test_loader_keeps_distinct_lines_and_drops_file_copies
PASS test_loschmidt_number_density
PASS test_low_abundance_and_short_path_limit
PASS test_optically_thick_limit
PASS test_ozone_column_in_dobson_units
PASS test_planck_peak_follows_wien_law_in_wavenumber
PASS 

## 7. Load data and set conditions

The following cell loads the ozone line list and defines the atmospheric reference conditions used for the temperature sweep.

In [10]:
O3, rep_o3, files_o3 = load_linelist("O3", 3, cfg.DATA_DIR)
print("O3 lines: %d | parameters replaced by data-derived estimates: %s" % (len(O3.nu), rep_o3))
for msg in dilution_warnings(cfg.X_O3): print(msg)
GRID = np.arange(cfg.NU_MIN, cfg.NU_MAX + 0.5 * cfg.NU_STEP, cfg.NU_STEP)
print("grid points:", len(GRID))
print("\n height  T (K)   P (hPa)")
for h in (0, 5, 10, 15, 20, 25, 30, 32):
    T_, P_ = isa_atmosphere(h * 1000.0); print(" %4d km %6.1f %9.1f" % (h, T_, P_ / 100.0))
print(" (about 1 hPa near 48 km, the top of the stratosphere)")

O3 lines: 25776 | parameters replaced by data-derived estimates: {'gamma_air': 587, 'n_air': 0, 'gamma_self': 0, 'duplicate_lines_removed': 0}
grid points: 50001

 height  T (K)   P (hPa)
    0 km  288.1    1013.2
    5 km  255.6     540.2
   10 km  223.1     264.4
   15 km  216.7     120.4
   20 km  216.7      54.7
   25 km  221.7      25.1
   30 km  226.7      11.7
   32 km  228.7       8.7
 (about 1 hPa near 48 km, the top of the stratosphere)


In [11]:
SWEEP_NAME = "temperature T (K) at fixed path length"

def build_conditions(T_ref, P_ref_pa, L_fix_m, n=None):
    # (T, P, L) for every frame. Only T is swept; L is constant; P follows T according to cfg.P_COUPLING.
    n = cfg.N_FRAMES if n is None else n
    Ts = np.linspace(cfg.T_RANGE[0], cfg.T_RANGE[1], n)
    if Ts[0] < T_ref < Ts[-1]:
        Ts[int(np.argmin(np.abs(Ts - T_ref)))] = T_ref     # the reference temperature is one of the frames, exactly
    if cfg.P_COUPLING == "const_density":
        Ps = P_ref_pa * Ts / T_ref                 # ideal gas, P = n k_B T with the number density n held fixed
    elif cfg.P_COUPLING == "const_pressure":
        Ps = np.full(n, float(P_ref_pa))
    else:
        raise ValueError("cfg.P_COUPLING must be 'const_density' or 'const_pressure'")
    return Ts, Ps, np.full(n, float(L_fix_m))

def compute_frames(Ts, Ps, Ls, x):
    # alpha_O3(nu) [cm^-1] for every frame. Exact on 'anchor' temperatures, cubic spline of ln(alpha) in between.
    t0 = time.time(); n = len(Ts)
    exact = np.arange(n) if cfg.EXACT_ALL else choose_anchors(Ts, Ps, dT_max=cfg.ANCHOR_DT_K)
    A = []
    for j, i in enumerate(exact):
        A.append(absorption_coefficient(GRID, O3, Ts[i], Ps[i], x, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN))
        print("  exact spectrum %2d/%d  T = %6.2f K  P = %8.2f Pa   (%.0f s)" % (j + 1, len(exact), Ts[i], Ps[i], time.time() - t0))
    A = np.array(A)
    full = A if len(exact) == n else interpolate_between_anchors(Ts, exact, A)
    if len(exact) < n:                       # accuracy check of the interpolation on one frame that is NOT an anchor
        rest = np.setdiff1d(np.arange(n), exact); mid = int(rest[len(rest) // 2])
        ex = absorption_coefficient(GRID, O3, Ts[mid], Ps[mid], x, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=cfg.S_MIN)
        core = ex > 1e-3 * ex.max()
        rel = float(np.max(np.abs(full[mid] - ex)[core] / ex[core]))
        itau_err = float(abs(trapezoid(full[mid] - ex, GRID)) / trapezoid(ex, GRID))
        print("interpolation check at T = %.2f K: max relative error of alpha %.2e, error of the band integral %.2e" % (Ts[mid], rel, itau_err))
        if rel > 5e-2: print("WARNING: interpolation error is large; set cfg.EXACT_ALL = True or reduce cfg.ANCHOR_DT_K.")
        full[mid] = ex
    return [dict(T=Ts[i], P=Ps[i], L=Ls[i], a=full[i]) for i in range(n)]

def pick_strongest_line(lines, nu_min, nu_max):
    inside = np.where((lines.nu >= nu_min) & (lines.nu <= nu_max))[0]
    return int(inside[np.argmax(lines.S[inside])])

def line_state(k, T, P_pa, x):
    # Everything about line k at (T, P): centre, strength, widths, number density of O3.
    S_T = float(line_strength_vs_T(O3, k, [T], hapi_partition_fn)[0])
    gD = float(doppler_hwhm(O3.nu[k], T, O3.mass_amu[int(O3.iso[k])]))
    gL = float(lorentz_hwhm(O3.gamma_air[k], O3.gamma_self[k], O3.n_air[k], T, P_pa, x))
    return dict(nu_c=float(O3.nu[k] + O3.delta_air[k] * P_pa / ATM_PA), S=S_T, gD=gD, gL=gL, gV=float(voigt_hwhm(gD, gL)),
                n_o3=x * float(number_density_cm3(T, P_pa)))

SOURCES = {"sun": 5772.0, "lamp": 1500.0, "earth": 288.0}
def make_I0(grid, src):
    if src == "flat":
        return np.ones_like(grid), "arbitrary units (flat: I0 = 1)"
    T = SOURCES[src]
    return planck_per_cm1(grid, T), "W m$^{-2}$ sr$^{-1}$ (cm$^{-1}$)$^{-1}$  (blackbody %.0f K)" % T

def export_csv(path, I0, frames, x, notes=""):
    # Long-format CSV, one row per (frame, wavenumber). Comment lines start with '#'.
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    cols = ["T_K", "P_Pa", "L_m", "wavenumber_cm-1", "wavelength_um", "I0", "alpha_O3_cm-1", "tau", "transmittance", "I_transmitted"]
    with open(path, "w") as fh:
        fh.write("# O3 transmission model (HITRAN lines, Voigt, Beer-Lambert); ILLUSTRATIVE, not a validated product\n")
        fh.write("# x_O3=%g sweep=%s %s\n" % (x, cfg.MODE, notes))
        fh.write(",".join(cols) + "\n")
        for fr in frames:
            tau, trans, I = beer_lambert(fr["a"], fr["L"] * 100.0, I0)
            np.savetxt(fh, np.column_stack([np.full_like(GRID, fr["T"]), np.full_like(GRID, fr["P"]), np.full_like(GRID, fr["L"]),
                                            GRID, 1.0e4 / GRID, I0, fr["a"], tau, trans, I]), delimiter=",", fmt="%.8g")
    print("Wrote", path)

def make_I0_um(lam_um, src):
    """Incident spectrum directly on the PLOTTED wavelength axis (per um).
    BUG FIX: the old code built I0 per cm^-1 and multiplied by the Jacobian 1e4/lambda^2 for every source,
    so even the 'flat' source (I0 = 1) became a curve ~ 1/lambda^2 (a 2 % tilt over this window).
    'flat' is now exactly 1 at every wavelength."""
    lam_um = np.asarray(lam_um, float)
    if src == "flat":
        return np.ones_like(lam_um), "Relative spectral intensity (flat source, I$_0$ = 1)"
    Tb = SOURCES[src]
    return planck_per_cm1(1.0e4 / lam_um, Tb) * 1.0e4 / lam_um ** 2, "Spectral radiance (W m$^{-2}$ sr$^{-1}$ µm$^{-1}$), blackbody %.0f K" % Tb

def pick_showcase_line(lines, nu_min, nu_max, margin=1.0, strong_frac=0.2, iso_cm1=0.03):
    """Best line to follow in panels B and C: among the lines at least `margin` cm-1 inside the window with S >= strong_frac * (strongest
    interior line), the one with the smallest total strength of neighbours within +/- iso_cm1, relative to its own strength.
    (The old rule 'strongest isolated line' found no isolated line in this dense band and fell back to the strongest line, which has a neighbour.)"""
    idx = np.where((lines.nu >= nu_min + margin) & (lines.nu <= nu_max - margin))[0]
    if len(idx) == 0: return pick_strongest_line(lines, nu_min, nu_max)
    cand = idx[lines.S[idx] >= strong_frac * lines.S[idx].max()]
    best, best_score = int(cand[0]), np.inf
    for k in cand:
        near = (np.abs(lines.nu - lines.nu[k]) < iso_cm1) & (np.arange(len(lines.nu)) != k)
        score = float(lines.S[near].sum() / lines.S[k])
        if score < best_score: best, best_score = int(k), score
    return best

def line_sweep(k):
    # Strength S, Doppler and Lorentz HWHM of line k at every frame of the sweep (used by the axis limits).
    S, gD, gL = [], [], []
    for fr in STATE["frames"]:
        s = line_state(k, fr["T"], fr["P"], STATE["x"]); S.append(s["S"]); gD.append(s["gD"]); gL.append(s["gL"])
    S, gD, gL = np.array(S), np.array(gD), np.array(gL)
    return dict(S=S, gD=gD, gL=gL, gV=voigt_hwhm(gD, gL))

In [12]:
def test_conditions_temperature_sweep():
    # only T is swept; L is constant; with P_COUPLING = "const_density" the number density n = P/(k_B T) is the same in every frame
    saved = (cfg.T_RANGE, cfg.P_COUPLING)
    try:
        cfg.T_RANGE = (190.0, 300.0)
        cfg.P_COUPLING = "const_density"
        Ts, Ps, Ls = build_conditions(227.65, 1008.0, 300.0, 11)
        assert len(Ts) == 11 and np.all(np.diff(Ts) > 0) and np.all(Ls == 300.0)
        n = Ps / (K_B * Ts)
        assert np.allclose(n, n[0], rtol=1e-12)
        assert np.allclose(Ps / Ts, 1008.0 / 227.65)               # passes through the reference state
        cfg.P_COUPLING = "const_pressure"
        Ts, Ps, Ls = build_conditions(227.65, 1008.0, 300.0, 11)
        assert np.all(Ps == 1008.0) and np.all(Ls == 300.0)
    finally:
        cfg.T_RANGE, cfg.P_COUPLING = saved

def test_widths_and_strength_trends_at_constant_density():
    # Doppler HWHM ~ sqrt(T); Lorentz HWHM ~ P (296/T)^n ~ T^(1-n) at constant density; low-E'' lines get STRONGER when T falls.
    T1, T2, n_air = 200.0, 300.0, 0.7
    d1, d2 = doppler_hwhm(1045.0, T1, 48.0), doppler_hwhm(1045.0, T2, 48.0)
    assert abs(d2 / d1 - np.sqrt(T2 / T1)) < 1e-12
    P1, P2 = 1000.0 * T1 / 227.65, 1000.0 * T2 / 227.65
    l1 = lorentz_hwhm(0.08, 0.08, n_air, T1, P1, 8e-6); l2 = lorentz_hwhm(0.08, 0.08, n_air, T2, P2, 8e-6)
    assert abs(l2 / l1 - (T2 / T1) ** (1.0 - n_air)) < 1e-9
    L_low = make_lines([1045.0], 1e-20, E=40.0); L_high = make_lines([1045.0], 1e-20, E=600.0)
    s_low = line_strength_vs_T(L_low, 0, [200.0, 300.0], toy_Q); s_high = line_strength_vs_T(L_high, 0, [200.0, 300.0], toy_Q)
    assert s_low[0] > s_low[1] and s_high[0] < s_high[1]

test_conditions_temperature_sweep(); print("PASS test_conditions_temperature_sweep")
test_widths_and_strength_trends_at_constant_density(); print("PASS test_widths_and_strength_trends_at_constant_density")

PASS test_conditions_temperature_sweep
PASS test_widths_and_strength_trends_at_constant_density


## 8. Interactive temperature sweep

Use **Play** or move the slider to step through the temperature frames. Change the path length, reference state, temperature range, ozone mole fraction or pressure rule using the controls, then select **Recompute**. Choose a spectral line using the line-centre field.

- **A:** Transmittance spectrum across the full wavenumber window.
- **B:** Transmittance near the selected line, comparing the full model with Gaussian, Lorentzian and Voigt line shapes.
- **C:** Optical depth of the selected line on a logarithmic scale; the horizontal reference marks $\tau=1$.

The figure title reports the current temperature, pressure, number density, path length, ozone column, line strength and broadening widths. If the widgets do not render, use `draw(idx, "flat", line_nu)`.

In [13]:
STATE = {}
T_FIX0 = float(cfg.T_FIX); P_FIX0 = cfg.P_FIX_HPA * 100.0; L_FIX0 = float(cfg.L_FIX_M)

def recompute(T_ref=None, P_ref_pa=None, L_fix_m=None, x=None):
    # Compute alpha(nu) for every temperature of the sweep (the slow step) and all quantities derived from it.
    T_ref = T_FIX0 if T_ref is None else T_ref
    P_ref_pa = P_FIX0 if P_ref_pa is None else P_ref_pa
    L_fix_m = L_FIX0 if L_fix_m is None else L_fix_m
    x = cfg.X_O3 if x is None else x
    for msg in dilution_warnings(x): print(msg)
    Ts, Ps, Ls = build_conditions(T_ref, P_ref_pa, L_fix_m)
    k = pick_showcase_line(O3, GRID[0], GRID[-1])
    gV_all = [voigt_hwhm(doppler_hwhm(O3.nu[k], T, O3.mass_amu[int(O3.iso[k])]),
                         lorentz_hwhm(O3.gamma_air[k], O3.gamma_self[k], O3.n_air[k], T, P, x)) for T, P in zip(Ts, Ps)]
    w = step_warning(cfg.NU_STEP, min(gV_all))
    if w: print(w)
    print("Preparing %d frames (T = %.0f ... %.0f K) on %d grid points; coupling: %s" % (len(Ts), Ts[0], Ts[-1], len(GRID), cfg.P_COUPLING))
    STATE.clear()
    STATE.update(Ts=Ts, Ps=Ps, Ls=Ls, sweep=Ts, x=x, k_default=k, T_ref=T_ref, T_fix=T_ref, P_fix=P_ref_pa, L_fix=L_fix_m, token=time.time())
    frames = compute_frames(Ts, Ps, Ls, x)
    L_cm = L_fix_m * 100.0
    trans = [np.exp(-f["a"] * L_cm) for f in frames]
    ref_idx = int(np.argmin(np.abs(Ts - T_ref)))
    span = GRID[-1] - GRID[0]
    STATE.update(frames=frames, ref_idx=ref_idx,
                 meanT=np.array([trapezoid(t, GRID) / span for t in trans]),
                 itau=np.array([integrated_tau(GRID, f["a"], L_cm) for f in frames]),
                 Tmin=float(min(t.min() for t in trans)), linesweep={})
    lo = max(0.0, np.floor((STATE["Tmin"] - 0.03) * 20.0) / 20.0)
    STATE["A_lo"] = lo
    s = line_state(k, Ts[ref_idx], Ps[ref_idx], x)
    tau0 = s["n_o3"] * s["S"] * float(voigt_profile(0.0, s["gD"], s["gL"])) * L_cm
    n_tot = float(number_density_cm3(Ts[ref_idx], Ps[ref_idx]))
    print("conditions: x_O3 = %.2f ppmv, L = %.4g m, n_total(T_ref) = %.3g cm^-3, n_O3 = %.3g cm^-3, column = %.3g DU"
          % (x * 1e6, L_fix_m, n_tot, x * n_tot, column_DU(x, Ts[ref_idx], Ps[ref_idx], L_fix_m)))
    print("selected line %.4f cm-1 (S = %.2e, E'' = %.1f cm-1): centre optical depth at T_ref = %.3g" % (O3.nu[k], O3.S[k], O3.E_low[k], tau0))
    print("mean transmittance of the window: %.3f (T = %.0f K) ... %.3f (T = %.0f K); darkest point of any spectrum: %.3f"
          % (STATE["meanT"][0], Ts[0], STATE["meanT"][-1], Ts[-1], STATE["Tmin"]))
    if STATE["meanT"].min() < 0.5:
        print("WARNING: more than half of the window is black at some temperature (very saturated). Reduce L (cfg.L_FIX_M) so that tau0 of the strongest line is about 1.")
    print("done.")

recompute()

Preparing 61 frames (T = 190 ... 300 K) on 50001 grid points; coupling: const_density
  exact spectrum  1/13  T = 190.00 K  P =   841.29 Pa   (18 s)
  exact spectrum  2/13  T = 199.17 K  P =   881.88 Pa   (36 s)
  exact spectrum  3/13  T = 208.33 K  P =   922.47 Pa   (55 s)
  exact spectrum  4/13  T = 217.50 K  P =   963.06 Pa   (73 s)
  exact spectrum  5/13  T = 226.67 K  P =  1003.65 Pa   (92 s)
  exact spectrum  6/13  T = 235.83 K  P =  1044.23 Pa   (110 s)
  exact spectrum  7/13  T = 245.00 K  P =  1084.82 Pa   (129 s)
  exact spectrum  8/13  T = 254.17 K  P =  1125.41 Pa   (148 s)
  exact spectrum  9/13  T = 263.33 K  P =  1166.00 Pa   (167 s)
  exact spectrum 10/13  T = 272.50 K  P =  1206.59 Pa   (186 s)
  exact spectrum 11/13  T = 281.67 K  P =  1247.18 Pa   (205 s)
  exact spectrum 12/13  T = 290.83 K  P =  1287.77 Pa   (225 s)
  exact spectrum 13/13  T = 300.00 K  P =  1328.35 Pa   (244 s)
interpolation check at T = 246.83 K: max relative error of alpha 1.14e-06, error of the

In [32]:
class Dashboard:
    # One persistent figure; update() changes only the data of existing artists (fast).
    #   A  (wide, top)    transmittance spectrum of the whole window at the current T, with the reference-T spectrum in grey behind it
    #   B  (middle left)  zoom on the selected line: full model and the three single-line shapes (+ reference T)
    #   C  (middle right) optical depth tau = n S phi L of that line (log axis; tau = 1 marked) (+ reference T)
    N_BINS = 1800
    SHAPE_STYLE = {"Gaussian": dict(color="C2", ls="--"), "Lorentzian": dict(color="C1", ls="-."), "Voigt": dict(color="C3", ls="-")}
    REF = dict(color="0.45", lw=1.2, ls=":")

    def __init__(self, figsize=(13.0, 9.3), dpi=None):
        dpi = cfg.FIG_DPI_SCREEN if dpi is None else dpi
        self.fig = plt.figure(figsize=figsize, dpi=dpi); plt.close(self.fig)
        gs = self.fig.add_gridspec(2, 2, height_ratios=[1.35, 1.0], left=0.075, right=0.935, top=0.835, bottom=0.095, hspace=0.62, wspace=0.26)
        self.axA = self.fig.add_subplot(gs[0, :]); self.axB = self.fig.add_subplot(gs[1, 0])
        self.axC = self.fig.add_subplot(gs[1, 1])
        self.title = self.fig.suptitle("", fontsize=10.5, y=0.985, va="top")
        self.k = None
        self.wavelength_um = 1.0e4 / GRID
        self.order = np.argsort(self.wavelength_um); self.wl = self.wavelength_um[self.order]
        one = np.ones_like(self.wl)
        # ---- A
        a = self.axA
        (self.lI0,) = a.plot(self.wl[[0, -1]], [1, 1], color="k", lw=1.0, ls="--", label="incident spectrum I$_0$")
        (self.lRef,) = a.plot(self.wl, one, lw=3.0, color="0.78", solid_capstyle="butt", label="reference T")
        (self.lT,) = a.plot(self.wl, one, lw=0.9, color="C3", zorder=3, label="current T")
        a.set_xlim(self.wl[0], self.wl[-1]); a.set_xlabel("Wavelength, λ (µm)", fontsize=10)
        self.legA = a.legend(fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, frameon=False)
        sec = a.secondary_xaxis("top", functions=(lambda l: 1.0e4 / np.where(l == 0, np.nan, l), lambda n: 1.0e4 / np.where(n == 0, np.nan, n)))
        sec.set_xlabel("Wavenumber, ν̃ (cm⁻¹)", fontsize=9); sec.tick_params(labelsize=8)
        self.vA = a.axvline(self.wl[0], color="g", ls=":", lw=1.3)
        # ---- B
        b = self.axB
        (self.bRef,) = b.plot([], [], label="Voigt, reference T", **self.REF)
        (self.bFull,) = b.plot([], [], color="k", lw=1.8, label="full model (all lines)")
        self.bShape = {n: b.plot([], [], lw=1.5, label="this line alone: " + n, **st)[0] for n, st in self.SHAPE_STYLE.items()}
        b.set_xlabel("Wavenumber offset, Δν̃ (cm⁻¹)", fontsize=10); b.set_ylabel("Transmittance", fontsize=10)
        b.legend(fontsize=8, loc="lower right", framealpha=0.9)
        # ---- C
        c = self.axC
        (self.cRef,) = c.semilogy([], [], label="Voigt, reference T", **self.REF)
        self.cShape = {n: c.semilogy([], [], lw=1.6, label=n, **st)[0] for n, st in self.SHAPE_STYLE.items()}
        c.axhline(1.0, color="k", ls=":", lw=1); c.text(0.99, 1.15, "τ = 1", transform=c.get_yaxis_transform(), ha="right", fontsize=8)
        c.set_xlabel("Wavenumber offset, Δν̃ (cm⁻¹)", fontsize=10); c.set_ylabel("Optical depth of this line, τ = n·S·φ·L", fontsize=10)
        c.legend(fontsize=8, loc="upper left", framealpha=0.9)
        
        for ax_, t in zip((self.axA, self.axB, self.axC),
                          ("A. Transmitted spectrum of the whole window (thick grey: at the reference temperature, red: at the current temperature)",
                           "B. Transmittance near the selected line: line-shape comparison",
                           "C. Optical depth τ of the selected line (Gaussian, Lorentzian, Voigt)")):
            ax_.set_title(t, fontsize=11, pad=36 if ax_ is self.axA else 7)
        for ax_ in (self.axA, self.axB, self.axC): ax_.tick_params(labelsize=9)
        for ax_ in (self.axA, self.axB, self.axC): ax_.grid(alpha=0.25)
        self._cache = {}

    def _line_cache(self, k):
        # per-line quantities over the whole sweep; recomputed only when another line is selected
        if k not in STATE["linesweep"]: STATE["linesweep"][k] = line_sweep(k)
        ls = STATE["linesweep"][k]; r = STATE["ref_idx"]
        if self.k != k:
            self.k = k
            # fixed axis limits of B and C for this line (so the frames do not jump)
            self.hB = 14.0 * float(ls["gV"].max())
            a_f = STATE["x"] * np.array([number_density_cm3(f["T"], f["P"]) for f in STATE["frames"]]) * ls["S"]
            tau_top = float(np.max(a_f * np.array([gaussian_profile(0.0, g) for g in ls["gD"]]) * STATE["L_fix"] * 100.0))
            self.cTop = 3.0 * tau_top
        return ls

    def update(self, idx, src, line_nu):
        fr = STATE["frames"][idx]; x = STATE["x"]; r = STATE["ref_idx"]; fref = STATE["frames"][r]
        T, P, L_m = fr["T"], fr["P"], fr["L"]; L_cm = L_m * 100.0
        k = int(np.argmin(np.abs(O3.nu - line_nu)))
        ls = self._line_cache(k)
        tau, trans, _ = beer_lambert(fr["a"], L_cm, 1.0)
        trans_ref = np.exp(-fref["a"] * L_cm)
        I0, unit = make_I0_um(self.wl, src)
        xe, ye = envelope(self.wl, I0 * trans[self.order], self.N_BINS); self.lT.set_data(xe, ye)
        xr, yr = envelope(self.wl, I0 * trans_ref[self.order], self.N_BINS); self.lRef.set_data(xr, yr)
        self.lI0.set_data(self.wl[[0, -1]], [I0[0], I0[-1]])
        if src == "flat": self.axA.set_ylim(STATE["A_lo"], 1.02); self.axA.set_ylabel("Transmittance  I / I$_0$ (flat source)", fontsize=10)
        else: self.axA.set_ylim(0.0, 1.08 * float(I0.max())); self.axA.set_ylabel(unit, fontsize=8)
        self.legA.get_texts()[1].set_text("reference T = %.2f K" % fref["T"]); self.legA.get_texts()[2].set_text("current T = %.2f K" % T)
        s = line_state(k, T, P, x); s0 = line_state(k, fref["T"], fref["P"], x)
        lam = 1.0e4 / s["nu_c"]; self.vA.set_xdata([lam, lam])
        h = self.hB; d = np.linspace(-h, h, 1801)
        a_f = s["n_o3"] * s["S"]; a_f0 = s0["n_o3"] * s0["S"]
        phis = {"Gaussian": gaussian_profile(d, s["gD"]), "Lorentzian": lorentz_profile(d, s["gL"]), "Voigt": voigt_profile(d, s["gD"], s["gL"])}
        phi_ref = voigt_profile(d, s0["gD"], s0["gL"])
        for n, phi in phis.items():
            self.bShape[n].set_data(d, np.exp(-a_f * phi * L_cm)); self.cShape[n].set_data(d, np.maximum(a_f * phi * L_cm, 1e-12))
        self.bRef.set_data(d, np.exp(-a_f0 * phi_ref * L_cm)); self.cRef.set_data(d, np.maximum(a_f0 * phi_ref * L_cm, 1e-12))
        m = (GRID >= s["nu_c"] - h) & (GRID <= s["nu_c"] + h)
        self.bFull.set_data(GRID[m] - s["nu_c"], trans[m])
        self.axB.set_xlim(-h, h); self.axC.set_xlim(-h, h)
        self.axB.set_ylim(max(0.0, min(float(trans[m].min()), float(np.exp(-a_f * phis["Gaussian"].max() * L_cm))) - 0.05), 1.03)
        self.axC.set_ylim(1e-4, max(self.cTop, 3.0))
        du = column_DU(x, T, P, L_m); ratio = s["gL"] / s["gD"]
        regime = "Doppler (Gaussian) dominates" if ratio < 0.2 else ("Lorentz dominates" if ratio > 5 else "mixed: Voigt regime")
        tau0 = a_f * float(voigt_profile(0.0, s["gD"], s["gL"])) * L_cm
        n_tot = float(number_density_cm3(T, P))
        coupling = "gas density held fixed (P ∝ T)" if cfg.P_COUPLING == "const_density" else "pressure held fixed"
        self.title.set_text(
            "O₃ spectral interval: %.0f–%.0f cm⁻¹ (%.2f–%.2f µm) | Sweep: temperature T, path length fixed | %s | illustrative model using HITRAN line data\n"
            "T = %.2f K   P = %.2f hPa   n(air) = %.3g cm⁻³   L = %.4g m (= %.3g DU of ozone)   x_O3 = %.2f ppmv   |   mean transmittance of window = %.3f\n"
            "selected line %.4f cm⁻¹ (E″ = %.1f cm⁻¹):  S(T) = %.3e cm/molecule (%.2f × its value at T_ref),  centre τ₀ = %.3g\n"
            "HWHM  Doppler = %.2e,  Lorentz = %.2e,  Voigt = %.2e cm⁻¹  →  %s (γL/γD = %.2g)"
            % (GRID[0], GRID[-1], 1e4 / GRID[-1], 1e4 / GRID[0], coupling, T, P / 100.0, n_tot, L_m, du, x * 1e6, float(trans.mean()),
               s["nu_c"], O3.E_low[k], s["S"], s["S"] / s0["S"], tau0, s["gD"], s["gL"], s["gV"], regime, ratio))

def draw(idx=None, src="flat", line_nu=None):
    # static fallback without widgets: draw(), draw(30), draw(30, "flat", 1047.6)
    idx = int(STATE["ref_idx"]) if idx is None else int(idx)
    line_nu = float(O3.nu[STATE["k_default"]]) if line_nu is None else line_nu
    d = Dashboard(); d.update(idx, src, line_nu); display(d.fig)

In [34]:
# ---- Interactive controls ----
def _default_idx():
    return int(STATE["ref_idx"])

IDX0 = _default_idx()
play = widgets.Play(min=0, max=len(STATE["sweep"]) - 1, step=1, interval=250, value=IDX0, description="Play")
sweep_slider = widgets.IntSlider(value=IDX0, min=0, max=len(STATE["sweep"]) - 1, step=1, description="Frame / temperature",
    continuous_update=False, readout=False, layout=widgets.Layout(width="420px"), style={"description_width": "initial"})
sweep_value_label = widgets.Label()
src_dd = widgets.Dropdown(options=[("Flat (I₀ = 1; relative units)", "flat"), ("Sun blackbody (5772 K)", "sun"), ("Lamp blackbody (1500 K)", "lamp"), ("Earth-like blackbody (288 K)", "earth")], value=cfg.SOURCE, description="Incident source")
line_box = widgets.FloatText(value=round(float(O3.nu[STATE["k_default"]]), 4), description="Line centre (cm⁻¹)", style={"description_width": "initial"})
coupling_dd = widgets.Dropdown(options=[("constant gas density (P ∝ T)", "const_density"), ("constant pressure", "const_pressure")], value=cfg.P_COUPLING, description="Pressure rule", style={"description_width": "initial"})
Tmin_box = widgets.FloatText(value=cfg.T_RANGE[0], description="T from (K)", style={"description_width": "initial"}, layout=widgets.Layout(width="150px"))
Tmax_box = widgets.FloatText(value=cfg.T_RANGE[1], description="to (K)", style={"description_width": "initial"}, layout=widgets.Layout(width="130px"))
Tref_box = widgets.FloatText(value=round(STATE["T_ref"], 2), description="Reference T (K)", style={"description_width": "initial"})
Pref_box = widgets.FloatText(value=round(STATE["P_fix"] / 100.0, 3), description="P at reference T (hPa)", style={"description_width": "initial"})
L_box = widgets.FloatText(value=STATE["L_fix"], description="Fixed path length (m)", style={"description_width": "initial"})
x_box = widgets.FloatText(value=cfg.X_O3 * 1e6, description="O₃ mole fraction (ppmv)", style={"description_width": "initial"})
btn_recompute = widgets.Button(description="Recompute (slow)", button_style="warning", layout=widgets.Layout(width="170px"))
btn_csv = widgets.Button(description="Export frame (CSV)", layout=widgets.Layout(width="150px"))
btn_png = widgets.Button(description="Save figure (PNG, high dpi)", layout=widgets.Layout(width="200px"))
btn_html = widgets.Button(description="Save animation (HTML)", layout=widgets.Layout(width="180px"))
btn_gif = widgets.Button(description="Save animation (GIF)", layout=widgets.Layout(width="160px"))
btn_mp4 = widgets.Button(description="Save animation (MP4)", layout=widgets.Layout(width="160px"))
stride_box = widgets.IntText(value=2, description="Frame stride", style={"description_width": "initial"}, layout=widgets.Layout(width="170px"))
fps_box = widgets.IntText(value=10, description="Animation fps", layout=widgets.Layout(width="150px"))
out = widgets.Output(); status = widgets.Output()
widgets.link((play, "value"), (sweep_slider, "value"))
DASH = {"obj": None, "token": None}

def get_dash():
    if DASH["obj"] is None or DASH["token"] != STATE["token"]:
        DASH["obj"], DASH["token"] = Dashboard(), STATE["token"]
    return DASH["obj"]

def redraw(*_):
    idx = int(sweep_slider.value); fr = STATE["frames"][idx]
    sweep_value_label.value = "T = %.2f K,  P = %.3g hPa" % (fr["T"], fr["P"] / 100.0)
    with out:
        clear_output(wait=True)
        d = get_dash(); d.update(idx, src_dd.value, line_box.value); display(d.fig)

def on_recompute(_):
    with status:
        clear_output(); play.max = 0
        cfg.T_RANGE = (float(Tmin_box.value), float(Tmax_box.value)); cfg.P_COUPLING = coupling_dd.value
        recompute(Tref_box.value, Pref_box.value * 100.0, L_box.value, x_box.value * 1e-6)
        sweep_slider.max = len(STATE["sweep"]) - 1; play.max = len(STATE["sweep"]) - 1
        sweep_slider.value = _default_idx(); play.value = sweep_slider.value
        line_box.value = round(float(O3.nu[STATE["k_default"]]), 4)
    redraw()

def on_csv(_):
    fr = STATE["frames"][int(sweep_slider.value)]
    I0, _u = make_I0(GRID, src_dd.value)
    with status:
        export_csv(os.path.join("outputs", "o3_T%.1fK_P%.1fhPa_L%.0fm.csv" % (fr["T"], fr["P"] / 100.0, fr["L"])), I0, [fr], STATE["x"])

def save_png(path=None, dpi=None):
    d = get_dash(); d.update(int(sweep_slider.value), src_dd.value, line_box.value)
    os.makedirs("outputs", exist_ok=True)
    path = path or os.path.join("outputs", "o3_T_frame%03d.png" % int(sweep_slider.value))
    d.fig.savefig(path, dpi=cfg.FIG_DPI_SAVE if dpi is None else dpi, bbox_inches="tight"); print("Saved", path)

def build_animation(stride=2, fps=10, dpi=70, src=None, line_nu=None):
    import matplotlib.animation as animation
    src = src_dd.value if src is None else src; line_nu = line_box.value if line_nu is None else line_nu
    d = Dashboard(dpi=dpi)
    idxs = list(range(0, len(STATE["sweep"]), max(1, int(stride))))
    def _frame(k):
        d.update(idxs[k], src, line_nu); return []
    return animation.FuncAnimation(d.fig, _frame, frames=len(idxs), interval=1000.0 / fps, blit=False, repeat=True)

def save_animation(fmt="html", stride=2, fps=10, dpi=250, name=None):
    # fmt: "html" (self-contained player), "gif" (needs pillow) or "mp4" (needs ffmpeg). Files are written to outputs/.
    import matplotlib.animation as animation
    os.makedirs("outputs", exist_ok=True)
    base = os.path.join("outputs", name or "o3_T_sweep")
    if fmt == "mp4" and not animation.writers.is_available("ffmpeg"):
        print("ffmpeg was not found on this computer, so MP4 cannot be written. Use HTML or GIF, or install ffmpeg."); return None
    anim = build_animation(stride, fps, dpi)
    if fmt == "html":
        with open(base + ".html", "w", encoding="utf-8") as fh: fh.write(anim.to_jshtml(fps=fps))
    elif fmt == "gif":
        anim.save(base + ".gif", writer="pillow", fps=fps)
    elif fmt == "mp4":
        anim.save(base + ".mp4", writer="ffmpeg", fps=fps, dpi=dpi)
    else:
        raise ValueError("fmt must be html, gif or mp4")
    print("Saved %s.%s  (%d frames)" % (base, fmt, len(range(0, len(STATE["sweep"]), max(1, int(stride))))))
    return anim

def _saver(fmt):
    def _cb(_):
        with status:
            try: save_animation(fmt, stride_box.value, fps_box.value)
            except Exception as exc: print("Could not save %s: %r" % (fmt, exc))
    return _cb

sweep_slider.observe(redraw, names="value")
for w in (src_dd, line_box): w.observe(redraw, names="value")
btn_recompute.on_click(on_recompute); btn_csv.on_click(on_csv); btn_png.on_click(lambda _: save_png())
btn_html.on_click(_saver("html")); btn_gif.on_click(_saver("gif")); btn_mp4.on_click(_saver("mp4"))
rows = [widgets.HBox([play, sweep_slider, sweep_value_label]),
        widgets.HBox([src_dd, line_box, coupling_dd]),
        widgets.HBox([Tmin_box, Tmax_box, Tref_box, Pref_box, L_box, x_box, btn_recompute]),
        widgets.HBox([btn_csv, btn_png, btn_html, btn_gif, btn_mp4, stride_box, fps_box]), status, out]
display(widgets.VBox(rows))
redraw()

### Animation export

The animation cell can create an embedded player and save HTML, GIF or MP4 files in the `outputs/` folder. Set `SAVE` to the required formats before running the cell.

In [29]:
plt.rcParams["animation.embed_limit"] = 300
SAVE = []                                  # e.g. ["html", "gif", "mp4"]
anim = build_animation(stride=2, fps=10)
for f in SAVE: save_animation(f, stride=2, fps=5)
HTML(anim.to_jshtml(fps=10))

### High-resolution figures

`save_hires_set()` saves PNG figures at the configured export resolution (`cfg.FIG_DPI_SAVE`). The **Save figure** button exports the currently selected frame.

In [30]:
def save_hires_set(temps=(190.0, 210.0, 227.65, 250.0, 275.0, 300.0), src=None, line_nu=None, dpi=None, folder="outputs/hires"):
    # Save one high-resolution PNG per requested temperature (nearest frame). Works with or without the widgets.
    src = cfg.SOURCE if src is None else src
    line_nu = float(O3.nu[STATE["k_default"]]) if line_nu is None else line_nu
    dpi = cfg.FIG_DPI_SAVE if dpi is None else dpi
    os.makedirs(folder, exist_ok=True)
    d = Dashboard()
    paths = []
    for T in temps:
        idx = int(np.argmin(np.abs(STATE["Ts"] - T)))
        d.update(idx, src, line_nu)
        path = os.path.join(folder, "o3_T%06.2fK_L%gm.png" % (STATE["Ts"][idx], STATE["L_fix"]))
        d.fig.savefig(path, dpi=dpi, bbox_inches="tight"); paths.append(path)
        print("Saved", path)
    return paths

save_hires_set()        # <- run this cell to write the six figures at 300 dpi (about 3900 x 3750 pixels each)

Saved outputs/hires\o3_T190.00K_L300m.png
Saved outputs/hires\o3_T210.17K_L300m.png
Saved outputs/hires\o3_T227.65K_L300m.png
Saved outputs/hires\o3_T250.50K_L300m.png
Saved outputs/hires\o3_T274.33K_L300m.png
Saved outputs/hires\o3_T300.00K_L300m.png


['outputs/hires\\o3_T190.00K_L300m.png',
 'outputs/hires\\o3_T210.17K_L300m.png',
 'outputs/hires\\o3_T227.65K_L300m.png',
 'outputs/hires\\o3_T250.50K_L300m.png',
 'outputs/hires\\o3_T274.33K_L300m.png',
 'outputs/hires\\o3_T300.00K_L300m.png']

## 9. Export data

The **Export CSV** button saves the currently selected frame. The following cell demonstrates how to export multiple frames from the temperature sweep.

In [18]:
I0_csv, _u = make_I0(GRID, cfg.SOURCE)
frames_csv = STATE["frames"][::10]          # every 10th temperature (all 61 frames would give a multi-hundred-MB file)
export_csv("outputs/o3_T_selected_frames.csv", I0_csv, frames_csv, STATE["x"], notes="coupling=%s L=%gm" % (cfg.P_COUPLING, STATE["L_fix"]))

Wrote outputs/o3_T_selected_frames.csv


## 10. Validation and numerical checks

Before interpreting the spectra, check the following:

1. **Spectral window:** Confirm that the selected interval contains the intended O₃ lines.
2. **Optical depth:** Check the line-centre optical depth and the minimum transmittance. Strongly saturated line cores may hide temperature-dependent changes.
3. **Interpolation:** Inspect the reported interpolation error. Set `cfg.EXACT_ALL = True` if greater accuracy is required.
4. **Spectral grid:** Check the grid resolution relative to the narrowest Voigt width; use `sensitivity()` to test the effect of changing the grid step.
5. **Line-wing and weak-line cut-offs:** Check their influence using the sensitivity analysis.
6. **HAPI comparison:** Run `check_against_hapi()` to compare the calculated cross-section with HAPI's Voigt routine.
7. **Temperature trends:** Confirm that the width and line-strength trend tests pass.
8. **Line assignment:** Use `show_assignment()` to inspect the quantum numbers of the selected HITRAN transition.
9. **Line count:** Check the loader output for unexpectedly removed records.

In [19]:
def strongest_lines(lines, n=8, lo=None, hi=None):
    lo = GRID[0] if lo is None else lo; hi = GRID[-1] if hi is None else hi
    sel = np.where((lines.nu >= lo) & (lines.nu <= hi))[0]
    top = sel[np.argsort(lines.S[sel])[::-1][:n]]
    print(lines.name, ": %d lines in %.0f-%.0f cm-1" % (len(sel), lo, hi))
    for k in top:
        print("  nu = %9.4f cm-1   S = %.3e   E'' = %8.2f   gamma_air = %.4f   iso %d"
              % (lines.nu[k], lines.S[k], lines.E_low[k], lines.gamma_air[k], lines.iso[k]))

strongest_lines(O3)

O3 : 4525 lines in 1040-1050 cm-1
  nu = 1049.4350 cm-1   S = 3.506e-20   E'' =    39.75   gamma_air = 0.0799   iso 1
  nu = 1049.0322 cm-1   S = 3.358e-20   E'' =    30.11   gamma_air = 0.0826   iso 1
  nu = 1049.3631 cm-1   S = 3.278e-20   E'' =    34.25   gamma_air = 0.0837   iso 1
  nu = 1049.6472 cm-1   S = 3.250e-20   E'' =    50.30   gamma_air = 0.0827   iso 1
  nu = 1049.0297 cm-1   S = 3.012e-20   E'' =    42.88   gamma_air = 0.0827   iso 1
  nu = 1048.0695 cm-1   S = 2.996e-20   E'' =    25.93   gamma_air = 0.0829   iso 1
  nu = 1049.4479 cm-1   S = 2.854e-20   E'' =    66.01   gamma_air = 0.0832   iso 1
  nu = 1047.6010 cm-1   S = 2.787e-20   E'' =    17.60   gamma_air = 0.0837   iso 1


In [20]:
def sensitivity(T=None, P_pa=None, L_m=None, window=None):
    # How much do the numerical approximations matter inside `window`? (uses the fixed values of this notebook by default)
    T = STATE["T_fix"] if T is None else T
    P_pa = STATE["P_fix"] if P_pa is None else P_pa
    L_cm = (STATE["L_fix"] if L_m is None else L_m) * 100.0
    window = (cfg.NU_MIN, min(cfg.NU_MIN + 3.0, cfg.NU_MAX)) if window is None else window
    base = dict(step=cfg.NU_STEP, wing=cfg.WING_CM1, smin=cfg.S_MIN)
    variants = [("baseline", {}), ("wing 2 cm-1", dict(wing=2.0)), ("wing 10 cm-1", dict(wing=10.0)), ("wing 25 cm-1", dict(wing=25.0)),
                ("S_min x100", dict(smin=cfg.S_MIN * 100)), ("S_min /100", dict(smin=cfg.S_MIN / 100)),
                ("step x2", dict(step=cfg.NU_STEP * 2)), ("step /2", dict(step=cfg.NU_STEP / 2))]
    print("window %.1f-%.1f cm-1, T = %.2f K, P = %.1f Pa, L = %.3g m" % (window + (T, P_pa, L_cm / 100)))
    print("%-14s %14s %14s %14s" % ("variant", "int tau dnu", "mean T", "rel. change"))
    ref = None
    for name, ch in variants:
        p = dict(base, **ch); g = np.arange(window[0], window[1], p["step"])
        a = absorption_coefficient(g, O3, T, P_pa, STATE["x"], hapi_partition_fn, wing_cm1=p["wing"], s_min=p["smin"])
        itau = integrated_tau(g, a, L_cm); mT = trapezoid(np.exp(-a * L_cm), g) / (g[-1] - g[0])
        ref = itau if ref is None else ref
        print("%-14s %14.6e %14.6e %14.3e" % (name, itau, mT, (itau - ref) / ref))
# sensitivity()      # takes a few minutes; shrink `window` for speed

In [26]:
def check_against_hapi(nu_a=1040.0, nu_b=1042.0, T_list=(296.0, 221.55), P_atm=0.0248, x_self=8e-6):
    # Compare our cross section with HAPI's absorptionCoefficient_Voigt for the same lines (needs data/ files).
    # Cores should agree closely at 296 K; far wings and T != 296 K can differ because of cut-off and exponent conventions.
    import hapi
    hapi.db_begin(cfg.DATA_DIR)
    table = [os.path.splitext(os.path.basename(p))[0]
             for p in glob.glob(os.path.join(cfg.DATA_DIR, "O3*.data")) + glob.glob(os.path.join(cfg.DATA_DIR, "O3*.par"))][0]
    g = np.arange(nu_a, nu_b, cfg.NU_STEP)
    for T in T_list:
        mine = species_cross_section(g, O3, T, P_atm * ATM_PA, x_self, hapi_partition_fn, wing_cm1=cfg.WING_CM1, s_min=0.0)
        nu_h, sig_h = hapi.absorptionCoefficient_Voigt(SourceTables=[table], Environment={"T": T, "p": P_atm},
                       WavenumberRange=(nu_a, nu_b), WavenumberStep=cfg.NU_STEP, HITRAN_units=True,
                       Diluent={"air": 1.0 - x_self, "self": x_self})
        ref = np.interp(g, nu_h, sig_h)
        core = mine > 0.2 * mine.max()
        rel = np.abs(mine[core] - ref[core]) / ref[core]
        print("O3 T=%.2f K: core points %d, max |rel diff| = %.3e, median = %.3e" % (T, core.sum(), rel.max(), np.median(rel)))

check_against_hapi()   # check HAPI's signature first: hapi.getHelp(hapi.absorptionCoefficient_Voigt)

Using data

O3
                     Lines parsed: 25776
{'air': 0.999992, 'self': 8e-06}
16.147489 seconds elapsed for abscoef; nlines = 25776
O3 T=296.00 K: core points 691, max |rel diff| = 5.029e-03, median = 1.736e-03
{'air': 0.999992, 'self': 8e-06}
15.450748 seconds elapsed for abscoef; nlines = 25776
O3 T=221.55 K: core points 856, max |rel diff| = 5.520e-03, median = 2.124e-03


## 11. References

- HITRAN, [Definitions and units](https://hitran.org/docs/definitions-and-units/).
- Kochanov et al., “HITRAN Application Programming Interface (HAPI)…”, *Journal of Quantitative Spectroscopy and Radiative Transfer* **177**, 15–30 (2016). https://doi.org/10.1016/j.jqsrt.2016.03.005
- Gordon et al., “The HITRAN2024 molecular spectroscopic database”, *Journal of Quantitative Spectroscopy and Radiative Transfer* **353**, 109807 (2026). https://doi.org/10.1016/j.jqsrt.2026.109807
- Gamache et al., “Partition sums for molecules and their isotopologues for HITRAN2024”, *Journal of Quantitative Spectroscopy and Radiative Transfer* **345**, 109568 (2025). https://doi.org/10.1016/j.jqsrt.2025.109568
- *U.S. Standard Atmosphere, 1976*, U.S. Government Printing Office. [NASA copy](https://ntrs.nasa.gov/api/citations/19770009539/downloads/19770009539.pdf).
- Olivero, J. J. and Longbothum, R. L., “Empirical fits to the Voigt line width: a brief review”, *Journal of Quantitative Spectroscopy and Radiative Transfer* **17**, 233–236 (1977). Verify bibliographic details before formal citation.

**Model assumptions:** The ozone mole fraction of 8 ppmv is an assumed value, not a measured profile. The Sun option represents a blackbody spectral shape, not solar irradiance at Earth.

## Interpretation and limitations

- The temperature sweep isolates temperature dependence under the selected pressure-coupling rule; constant-density heating is an idealisation, not a vertical atmospheric profile.
- Synthetic-line tests verify selected numerical properties but do not establish agreement with measured ozone spectra.
- Confirm the HITRAN line list, spectral-window line count and HAPI comparison before using the spectra quantitatively.
- Wavelength conversion is $\lambda(\mu\mathrm m)=10^4/\tilde\nu(\mathrm{cm}^{-1})$. Spectral densities require the corresponding Jacobian when changing variables.